In [ ]:
# RELOAD — start of a new session: rebuild data + splits without re-preprocessing
import os, json, copy, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CK = "/kaggle/input/<your-dataset>/mtp_ckpt"     # or "/kaggle/working/mtp_ckpt" if you unzipped it there
meta = json.load(open(f"{CK}/meta.json"))
EX_ITEM, EX_SHARED, MAXN, SPLIT, MODES = meta["EX_ITEM"], meta["EX_SHARED"], meta["MAXN"], meta["SPLIT"], meta["MODES"]

L = np.load(f"{CK}/lpmc_data.npz")
masks = {"train": L["is_train"], "val": L["is_val"], "test": L["is_test"]}
S_year = {k: tuple(torch.tensor(a[m]).to(dev) for a in (L["X_item_s"], L["X_shared_s"], L["y"])) for k, m in masks.items()}
S = S_year

E = np.load(f"{CK}/expedia_data.npz")
ex_masks = {"train": E["is_tr"], "val": E["is_va"], "test": E["is_te"]}
ex_S = {k: tuple(torch.tensor(E[a][m]).to(dev) for a in ("Xi", "Xs", "pos", "mask", "y")) for k, m in ex_masks.items()}
rows = pd.read_csv(f"{CK}/lpmc_results.csv").to_dict("records")
ex_rows = pd.read_csv(f"{CK}/expedia_results.csv").to_dict("records")
print({k: len(v[-1]) for k, v in ex_S.items()}, "| models saved:", sorted(os.listdir(f"{CK}/models")))

In [1]:
# Cell 0 — fetch LPMC directly into Kaggle
!curl -sSIL http://transp-or.epfl.ch/data/lpmc.dat | grep -iE "^HTTP|content-length|location"
!wget -q -O /kaggle/working/lpmc.dat http://transp-or.epfl.ch/data/lpmc.dat
!ls -la /kaggle/working/lpmc.dat
!head -c 600 /kaggle/working/lpmc.dat

HTTP/1.1 301 Moved Permanently
Content-Length: 162
Location: https://transp-or.epfl.ch/data/lpmc.dat
HTTP/2 200 
content-length: 13652718
-rw-r--r-- 1 root root 13652718 Oct 24  2019 /kaggle/working/lpmc.dat
trip_id	household_id	person_n	trip_n	travel_mode	purpose	fueltype	faretype	bus_scale	survey_year	travel_year	travel_month	travel_date	day_of_week	start_time	age	female	driving_license	car_ownership	distance	dur_walking	dur_cycling	dur_pt_access	dur_pt_rail	dur_pt_bus	dur_pt_int	pt_interchanges	dur_driving	cost_transit	cost_driving_fuel	cost_driving_ccharge	driving_traffic_percent
0	0	0	0	4	3	1	1	1.0	1	2012	4	1	7	10.0	56	1	1	2	778	0.218055556	0.09222222199999999	0.134444444	0.0	0.016666667	0.0	0	0.052222222	1.5	0.14	0.0	0.111702
1	0	0	1	4	3	1	1	1.0	1	2012	4	1	7	13.0	56	1	1	2	778	0.18888888899

In [2]:
# Cell 1 — Load LPMC and look at its raw shape
import os, urllib.request
import numpy as np
import pandas as pd

DATA_URL = "http://transp-or.epfl.ch/data/lpmc.dat"
LOCAL    = "/kaggle/working/lpmc.dat"
KAGGLE_INPUT = None   # set this if you uploaded it, e.g. "/kaggle/input/lpmc/lpmc.dat"

if KAGGLE_INPUT and os.path.exists(KAGGLE_INPUT):
    path = KAGGLE_INPUT
else:
    if not os.path.exists(LOCAL):
        urllib.request.urlretrieve(DATA_URL, LOCAL)
    path = LOCAL

df = pd.read_csv(path, sep="\t")
if df.shape[1] == 1:                       # fallback if it isn't tab-separated
    df = pd.read_csv(path, sep=r"\s+")
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

print("file :", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
print("shape:", df.shape)
print("columns:", list(df.columns))
df.head(3).T

file : /kaggle/working/lpmc.dat (13.7 MB)
shape: (81086, 32)
columns: ['trip_id', 'household_id', 'person_n', 'trip_n', 'travel_mode', 'purpose', 'fueltype', 'faretype', 'bus_scale', 'survey_year', 'travel_year', 'travel_month', 'travel_date', 'day_of_week', 'start_time', 'age', 'female', 'driving_license', 'car_ownership', 'distance', 'dur_walking', 'dur_cycling', 'dur_pt_access', 'dur_pt_rail', 'dur_pt_bus', 'dur_pt_int', 'pt_interchanges', 'dur_driving', 'cost_transit', 'cost_driving_fuel', 'cost_driving_ccharge', 'driving_traffic_percent']


,0,1,2
trip_id,0.000000,1.000000,2.000000
household_id,0.000000,0.000000,0.000000
person_n,0.000000,0.000000,0.000000
trip_n,0.000000,1.000000,2.000000
travel_mode,4.000000,4.000000,4.000000
purpose,3.000000,3.000000,3.000000
fueltype,1.000000,1.000000,1.000000
faretype,1.000000,1.000000,1.000000
bus_scale,1.000000,1.000000,1.000000
survey_year,1.000000,1.000000,1.000000


In [3]:
# Cell 2 — What we will model: the choice, the year split, and whether all 4 modes are always available
print("travel_mode counts (1 walk, 2 cycle, 3 PT, 4 drive):")
print(df["travel_mode"].value_counts().sort_index().to_dict())
print("shares (%):", (df["travel_mode"].value_counts(normalize=True).sort_index()*100).round(2).to_dict(), "\n")

print("trips per survey_year:", df["survey_year"].value_counts().sort_index().to_dict())
cross = (df.groupby("household_id")["survey_year"].nunique() > 1).sum()
print("households:", df["household_id"].nunique(), "| households spanning >1 survey year:", cross, "\n")

m = df.isna().sum()
print("missing values:", m[m > 0].to_dict() if (m > 0).any() else "none", "\n")

los = ["dur_walking","dur_cycling","dur_pt_access","dur_pt_rail","dur_pt_bus","dur_pt_int",
       "pt_interchanges","dur_driving","cost_transit","cost_driving_fuel",
       "cost_driving_ccharge","driving_traffic_percent"]
print(df[los].describe().T[["min","50%","max"]].round(3), "\n")

pt_total = df[["dur_pt_access","dur_pt_rail","dur_pt_bus","dur_pt_int"]].sum(axis=1)
print("rows with zero duration for a mode (would mean 'not available'):")
print({"walk": int((df["dur_walking"] <= 0).sum()),
       "cycle": int((df["dur_cycling"] <= 0).sum()),
       "pt": int((pt_total <= 0).sum()),
       "drive": int((df["dur_driving"] <= 0).sum())})

travel_mode counts (1 walk, 2 cycle, 3 PT, 4 drive):
{1: 14268, 2: 2405, 3: 28605, 4: 35808}
shares (%): {1: 17.6, 2: 2.97, 3: 35.28, 4: 44.16} 

trips per survey_year: {1: 27478, 2: 27288, 3: 26320}
households: 17616 | households spanning >1 survey year: 0 

missing values: none 

                           min    50%     max
dur_walking              0.025  0.723   9.278
dur_cycling              0.006  0.232   3.052
dur_pt_access            0.000  0.144   1.189
dur_pt_rail              0.000  0.000   1.467
dur_pt_bus               0.000  0.112   2.147
dur_pt_int               0.000  0.000   0.865
pt_interchanges          0.000  0.000   4.000
dur_driving              0.000  0.192   2.061
cost_transit             0.000  1.500  13.490
cost_driving_fuel        0.000  0.540  10.090
cost_driving_ccharge     0.000  0.000  10.500
driving_traffic_percent  0.000  0.306   1.250 

rows with zero duration for a mode (would mean 'not available'):
{'walk': 0, 'cycle': 0, 'pt': 0, 'drive': 0}


In [4]:
# Cell 3 — DeepHalo-style inputs: 4 alternatives x 8 item features, plus 17 shared features
MODES = ["walk", "cycle", "pt", "drive"]          # travel_mode 1..4 -> index 0..3
N = len(df)
zeros = np.zeros(N)

pt_dur = df[["dur_pt_access", "dur_pt_rail", "dur_pt_bus", "dur_pt_int"]].sum(axis=1).values
duration     = np.stack([df["dur_walking"].values, df["dur_cycling"].values, pt_dur, df["dur_driving"].values], axis=1)
cost         = np.stack([zeros, zeros, df["cost_transit"].values,
                         (df["cost_driving_fuel"] + df["cost_driving_ccharge"]).values], axis=1)
interchanges = np.stack([zeros, zeros, df["pt_interchanges"].values, zeros], axis=1)
traffic      = np.stack([zeros, zeros, zeros, df["driving_traffic_percent"].values], axis=1)
mode_id      = np.broadcast_to(np.eye(4), (N, 4, 4))

X_item = np.concatenate([np.stack([duration, cost, interchanges, traffic], axis=2), mode_id],
                        axis=2).astype(np.float32)                                   # (N, 4, 8)

print("purpose values:", sorted(df["purpose"].unique()), "| day_of_week values:", sorted(df["day_of_week"].unique()))
num_shared = df[["distance", "age", "female", "driving_license", "car_ownership"]].values
dow  = pd.get_dummies(df["day_of_week"], prefix="dow", dtype=float).reindex(columns=[f"dow_{i}" for i in range(1, 8)], fill_value=0.0).values
purp = pd.get_dummies(df["purpose"], prefix="purpose", dtype=float).reindex(columns=[f"purpose_{i}" for i in range(1, 6)], fill_value=0.0).values
X_shared = np.concatenate([num_shared, dow, purp], axis=1).astype(np.float32)        # (N, 17)

y    = (df["travel_mode"].values - 1).astype(np.int64)
year = df["survey_year"].values
hh   = df["household_id"].values

print("X_item  :", X_item.shape, "(trips, alternatives, item features)")
print("X_shared:", X_shared.shape, "(trips, shared features)")
print("y counts:", dict(zip(MODES, np.bincount(y))))

attr = ["duration_h", "cost_gbp", "interchanges", "traffic"]
print("\nmean of each attribute per alternative:")
print(pd.DataFrame({m: X_item[:, j, :4].mean(axis=0).round(3) for j, m in enumerate(MODES)}, index=attr))

purpose values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] | day_of_week values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
X_item  : (81086, 4, 8) (trips, alternatives, item features)
X_shared: (81086, 17) (trips, shared features)
y counts: {'walk': np.int64(14268), 'cycle': np.int64(2405), 'pt': np.int64(28605), 'drive': np.int64(35808)}

mean of each attribute per alternative:
               walk  cycle     pt  drive
duration_h    1.129  0.362  0.466  0.282
cost_gbp      0.000  0.000  1.563  1.903
interchanges  0.000  0.000  0.369  0.000
traffic       0.000  0.000  0.000  0.336


In [5]:
# Cell 4 — Split (test = survey year 3; val = 10% of households from years 1–2), scale using train only
import torch
rng = np.random.default_rng(0)

trainval_hh = np.unique(hh[year <= 2])
val_hh = rng.choice(trainval_hh, size=int(0.10 * len(trainval_hh)), replace=False)

is_test  = year == 3
is_val   = (year <= 2) & np.isin(hh, val_hh)
is_train = (year <= 2) & ~is_val
assert not (is_train & is_val).any() and (is_train | is_val | is_test).all()

# item attributes: divide by train std only (no centring), so 0 still means "not applicable"
item_scale = X_item[is_train][:, :, :4].reshape(-1, 4).std(axis=0)
X_item_s = X_item.copy()
X_item_s[:, :, :4] /= item_scale

# shared numeric features (distance, age, car_ownership): z-score on train; binaries and one-hots untouched
num_idx = [0, 1, 4]
mu = X_shared[is_train][:, num_idx].mean(axis=0)
sd = X_shared[is_train][:, num_idx].std(axis=0)
X_shared_s = X_shared.copy()
X_shared_s[:, num_idx] = (X_shared[:, num_idx] - mu) / sd

masks = {"train": is_train, "val": is_val, "test": is_test}
splits = {k: (torch.tensor(X_item_s[m]), torch.tensor(X_shared_s[m]), torch.tensor(y[m])) for k, m in masks.items()}

for k, m in masks.items():
    shares = (np.bincount(y[m], minlength=4) / m.sum() * 100).round(2)
    print(f"{k:5s}: {m.sum():6d} trips | {len(np.unique(hh[m])):5d} households | shares % {dict(zip(MODES, shares))}")

# reference loss: predict the training mode shares for everyone (no features at all)
p_train = np.bincount(y[is_train], minlength=4) / is_train.sum()
for k in ["val", "test"]:
    print(f"share-only NLL on {k}: {-np.log(p_train[y[masks[k]]]).mean():.4f}")

print("\nitem scales (train std):", dict(zip(attr, item_scale.round(3))))
print("shared numeric mean:", dict(zip(["distance", "age", "car_ownership"], mu.round(2))),
      "| std:", dict(zip(["distance", "age", "car_ownership"], sd.round(2))))

train:  49327 trips | 10553 households | shares % {'walk': np.float64(17.62), 'cycle': np.float64(2.88), 'pt': np.float64(34.87), 'drive': np.float64(44.64)}
val  :   5439 trips |  1172 households | shares % {'walk': np.float64(16.44), 'cycle': np.float64(2.3), 'pt': np.float64(35.04), 'drive': np.float64(46.22)}
test :  26320 trips |  5891 households | shares % {'walk': np.float64(17.8), 'cycle': np.float64(3.27), 'pt': np.float64(36.1), 'drive': np.float64(42.83)}
share-only NLL on val: 1.1090
share-only NLL on test: 1.1509

item scales (train std): {'duration_h': np.float32(0.7), 'cost_gbp': np.float32(2.095), 'interchanges': np.float32(0.345), 'traffic': np.float32(0.176)}
shared numeric mean: {'distance': np.float32(4545.0), 'age': np.float32(39.61), 'car_ownership': np.float32(0.99)} | std: {'distance': np.float32(4764.89), 'age': np.float32(19.33), 'car_ownership': np.float32(0.75)}


In [6]:
# Cell 5 — Two multinomial logits (convex, full-batch L-BFGS): the floor for every later model
import torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
S = {k: tuple(t.to(dev) for t in v) for k, v in splits.items()}

class MNL_A(nn.Module):            # u_j = beta' x_j : 4 generic attribute coefficients + 4 ASCs (via the one-hot)
    def __init__(self):
        super().__init__(); self.beta = nn.Linear(8, 1, bias=False)
    def forward(self, xi, xs):
        return self.beta(xi).squeeze(-1)

class MNL_B(MNL_A):                # + mode-specific effects of the 17 shared features
    def __init__(self):
        super().__init__(); self.gamma = nn.Linear(17, 4, bias=False)
    def forward(self, xi, xs):
        return super().forward(xi, xs) + self.gamma(xs)

def nll_acc(model, split):
    xi, xs, yy = S[split]
    with torch.no_grad():
        logits = model(xi, xs)
        return F.cross_entropy(logits, yy).item(), (logits.argmax(1) == yy).float().mean().item()

def fit_lbfgs(model, max_iter=1000):
    xi, xs, yy = S["train"]
    opt = torch.optim.LBFGS(model.parameters(), lr=1, max_iter=max_iter, tolerance_grad=1e-9,
                            tolerance_change=1e-12, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(model(xi, xs), yy); loss.backward(); return loss
    opt.step(closure)
    opt.zero_grad(); F.cross_entropy(model(xi, xs), yy).backward()
    gnorm = torch.cat([p.grad.flatten() for p in model.parameters()]).norm().item()
    return model, gnorm

mnl = {}
for name, cls in [("MNL-A (attributes + ASCs)", MNL_A), ("MNL-B (+ mode-specific shared)", MNL_B)]:
    m, g = fit_lbfgs(cls().to(dev))
    mnl[name] = m
    row = {k: nll_acc(m, k) for k in ["train", "val", "test"]}
    print(f"{name:31s} " + " | ".join(f"{k} NLL {v[0]:.4f} acc {v[1]:.4f}" for k, v in row.items()) + f" | grad norm {g:.1e}")

print("\nMNL-A coefficients (per train-std unit; ASCs only meaningful relative to each other):")
print(dict(zip(attr + [f"asc_{m}" for m in MODES],
               mnl["MNL-A (attributes + ASCs)"].beta.weight.detach().cpu().numpy().ravel().round(3))))
print("\nDeepHalo paper — MNL: train 0.8813, val 0.8621, test 0.8637 | DeepHalo: test 0.6407")

MNL-A (attributes + ASCs)       train NLL 0.8731 acc 0.6366 | val NLL 0.8883 acc 0.6172 | test NLL 0.8843 acc 0.6312 | grad norm 5.8e-05
MNL-B (+ mode-specific shared)  train NLL 0.7005 acc 0.7291 | val NLL 0.7220 acc 0.7113 | test NLL 0.7242 acc 0.7229 | grad norm 3.5e-05

MNL-A coefficients (per train-std unit; ASCs only meaningful relative to each other):
{'duration_h': np.float32(-4.16), 'cost_gbp': np.float32(-0.253), 'interchanges': np.float32(0.224), 'traffic': np.float32(-0.518), 'asc_walk': np.float32(1.393), 'asc_cycle': np.float32(-2.573), 'asc_pt': np.float32(0.521), 'asc_drive': np.float32(0.891)}

DeepHalo paper — MNL: train 0.8813, val 0.8621, test 0.8637 | DeepHalo: test 0.6407


In [7]:
# Cell 5b — (1) Is the 0.8843 vs 0.8637 gap our split or our features? Refit MNL-A on a random 80/10/10 row split.
#           (2) Does the interchanges sign flip once shared features (distance etc.) are in?
perm = np.random.default_rng(1).permutation(N)
n_tr, n_va = int(0.8 * N), int(0.1 * N)
idx = {"train": perm[:n_tr], "val": perm[n_tr:n_tr + n_va], "test": perm[n_tr + n_va:]}

sc = X_item[idx["train"]][:, :, :4].reshape(-1, 4).std(axis=0)   # same scaling recipe as Cell 4
Xi_r = X_item.copy(); Xi_r[:, :, :4] /= sc

S_year = S
S = {k: (torch.tensor(Xi_r[v]).to(dev), torch.tensor(X_shared_s[v]).to(dev), torch.tensor(y[v]).to(dev))
     for k, v in idx.items()}
m, g = fit_lbfgs(MNL_A().to(dev))
print("MNL-A, random row split: " + " | ".join(f"{k} NLL {nll_acc(m, k)[0]:.4f}" for k in ["train", "val", "test"])
      + f" | grad norm {g:.1e}")
print("paper:                   train 0.8813 | val 0.8621 | test 0.8637")
S = S_year   # restore the year-based split for everything that follows

print("\nMNL-B attribute coefficients:",
      dict(zip(attr, mnl["MNL-B (+ mode-specific shared)"].beta.weight.detach().cpu().numpy().ravel()[:4].round(3))))

MNL-A, random row split: train NLL 0.8776 | val NLL 0.8843 | test NLL 0.8704 | grad norm 1.2e-05
paper:                   train 0.8813 | val 0.8621 | test 0.8637

MNL-B attribute coefficients: {'duration_h': np.float32(-2.183), 'cost_gbp': np.float32(-0.242), 'interchanges': np.float32(-0.007), 'traffic': np.float32(-0.534)}


In [8]:
# Cell 6 — DeepHalo, copied unchanged from the authors' repo (DeepHalo-Code/Model/FeatureBased.py)
def make_valid_mask(n, lengths, device, mode='without'):
    row = torch.arange(n, device=device).unsqueeze(0)
    if mode == 'with':
        return (row < lengths.unsqueeze(1)) | (row == n - 1)
    else:
        return row < lengths.unsqueeze(1)

def masked_softmax(scores, lengths):
    n = scores.size(1)
    valid = make_valid_mask(n, lengths, scores.device)
    scores = scores.masked_fill(~valid, float('-inf'))
    return F.log_softmax(scores, dim=-1)

class NonlinearTransformation(nn.Module):
    def __init__(self, H, embed=128, dropout=0):
        super().__init__()
        self.fc1 = nn.Linear(embed, embed * H)
        self.fc2 = nn.Linear(embed, embed)
        self.H = H; self.embed = embed
        self.dropout = nn.Dropout(dropout)
        self.enc_norm = nn.LayerNorm(embed)
    def forward(self, X):
        B, n, _ = X.shape
        X = self.fc1(X).view(B, n, self.H, self.embed)
        X = nn.ReLU()(X)
        X = self.dropout(X)
        X = self.fc2(X)
        X = self.enc_norm(X)
        return X

class DeepHalo(nn.Module):
    def __init__(self, n, input_dim, H, L, embed=128, dropout=0):
        super().__init__()
        self.basic_encoder = nn.Sequential(
            nn.Linear(input_dim, embed), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed, embed), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed, embed))
        self.enc_norm = nn.LayerNorm(embed)
        self.aggregate_linear = nn.ModuleList([nn.Linear(embed, H) for _ in range(L)])
        self.nonlinear = nn.ModuleList([NonlinearTransformation(H, embed) for _ in range(L)])
        self.H = H; self.embed = embed
        self.final_linear = nn.Linear(embed, 1)
        self.qualinear1 = nn.Linear(embed, embed)   # defined but unused in the released code
        self.qualinear2 = nn.Linear(embed, embed)
    def forward(self, X, lengths):
        B, n, _ = X.shape
        Z = self.enc_norm(self.basic_encoder(X))
        X = Z.clone()
        for fc, nt in zip(self.aggregate_linear, self.nonlinear):
            Z_bar = (fc(Z).sum(1) / lengths.unsqueeze(1)).unsqueeze(-1).unsqueeze(1)
            phi = nt(X)
            valid = make_valid_mask(n, lengths, X.device)
            phi = phi * valid.unsqueeze(-1).unsqueeze(-1)
            Z = (phi * Z_bar).sum(2) / self.H + Z
        logits = self.final_linear(Z).squeeze(-1)
        probs = masked_softmax(logits, lengths)
        return probs, logits

# --- thin wrapper so every model has the same interface: model(x_item, x_shared) -> logits (B, 4)
def make_input(xi, xs):                 # each alternative sees [its 8 item features | the 17 shared features]
    return torch.cat([xi, xs.unsqueeze(1).expand(-1, xi.shape[1], -1)], dim=-1)

class DeepHaloChoice(nn.Module):
    def __init__(self, **kw):
        super().__init__(); self.net = DeepHalo(**kw)
    def forward(self, xi, xs):
        X = make_input(xi, xs)
        lengths = torch.full((X.shape[0],), X.shape[1], device=X.device)
        return self.net(X, lengths)[1]

torch.manual_seed(0)
_m = DeepHaloChoice(n=4, input_dim=25, H=8, L=4, embed=32).to(dev)
xi, xs, yy = S["train"]
print("params:", sum(p.numel() for p in _m.parameters() if p.requires_grad))
print("forward check:", _m(xi[:5], xs[:5]).shape, "(should be [5, 4])")

params: 44481
forward check: torch.Size([5, 4]) (should be [5, 4])


In [9]:
# Cell 7 — Reusable trainer (DeepHalo's LPMC recipe: Adam, batch 256, lr 1e-3 then 1e-4, patience 10) + one run
import time, copy

def evaluate(model, split, bs=8192):
    xi, xs, yy = S[split]; model.eval(); nll = 0.0; correct = 0
    with torch.no_grad():
        for i in range(0, len(yy), bs):
            lg = model(xi[i:i+bs], xs[i:i+bs])
            nll += F.cross_entropy(lg, yy[i:i+bs], reduction="sum").item()
            correct += (lg.argmax(1) == yy[i:i+bs]).sum().item()
    return nll / len(yy), correct / len(yy)

def train_model(model, phases=((1e-3, 10), (1e-4, 10)), bs=256, max_epochs=300, seed=0):
    g = torch.Generator().manual_seed(seed)
    xi, xs, yy = S["train"]; n = len(yy)
    best_v, best_state = evaluate(model, "val")[0], copy.deepcopy(model.state_dict())
    epochs_run = []
    for lr, patience in phases:
        model.load_state_dict(best_state)
        opt = torch.optim.Adam(model.parameters(), lr=lr); wait = 0
        for ep in range(max_epochs):
            model.train()
            perm = torch.randperm(n, generator=g).to(dev)
            for i in range(0, n, bs):
                b = perm[i:i+bs]
                loss = F.cross_entropy(model(xi[b], xs[b]), yy[b])
                opt.zero_grad(); loss.backward(); opt.step()
            v = evaluate(model, "val")[0]
            if v < best_v - 1e-5:
                best_v, best_state, wait = v, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
                if wait >= patience: break
        epochs_run.append(ep + 1)
    model.load_state_dict(best_state)
    return model, epochs_run

seed = 0
torch.manual_seed(seed)
dh = DeepHaloChoice(n=4, input_dim=25, H=8, L=4, embed=32).to(dev)
t0 = time.time()
dh, ep = train_model(dh, seed=seed)
print(f"DeepHalo seed {seed}: epochs per phase {ep} | {time.time() - t0:.0f} s")
for k in ["train", "val", "test"]:
    nll, acc = evaluate(dh, k)
    print(f"  {k:5s} NLL {nll:.4f} | acc {acc:.4f}")
print("our split so far: MNL-A test 0.8843 | MNL-B test 0.7242   (paper, random split: DeepHalo 0.6407)")

DeepHalo seed 0: epochs per phase [18, 10] | 37 s
  train NLL 0.6284 | acc 0.7561
  val   NLL 0.6699 | acc 0.7384
  test  NLL 0.6880 | acc 0.7366
our split so far: MNL-A test 0.8843 | MNL-B test 0.7242   (paper, random split: DeepHalo 0.6407)


In [10]:
# Cell 8 — Reproduction check: DeepHalo on the random 80/10/10 row split from Cell 5b (the paper's likely protocol)
# (random rows leak households across splits — used ONLY to compare with the paper's 0.6407)
mu_r = X_shared[idx["train"]][:, num_idx].mean(axis=0)
sd_r = X_shared[idx["train"]][:, num_idx].std(axis=0)
Xs_r = X_shared.copy(); Xs_r[:, num_idx] = (X_shared[:, num_idx] - mu_r) / sd_r

S_year = S
S = {k: (torch.tensor(Xi_r[v]).to(dev), torch.tensor(Xs_r[v]).to(dev), torch.tensor(y[v]).to(dev))
     for k, v in idx.items()}
torch.manual_seed(0)
dh_r, ep = train_model(DeepHaloChoice(n=4, input_dim=25, H=8, L=4, embed=32).to(dev), seed=0)
print(f"DeepHalo, random split: epochs {ep} | " +
      " | ".join(f"{k} NLL {evaluate(dh_r, k)[0]:.4f}" for k in ["train", "val", "test"]))
print("paper:                  train 0.6427 | val 0.6412 | test 0.6407")
S = S_year   # back to the year-based split

DeepHalo, random split: epochs [30, 12] | train NLL 0.5822 | val NLL 0.6367 | test NLL 0.6403
paper:                  train 0.6427 | val 0.6412 | test 0.6407


In [11]:
# Cell 9 — Context-free neural baseline: u_j = f([x_j | s]). Each mode sees only its own features + the traveller.
#          IIA holds by construction, so DeepHalo minus this = the value of cross-mode context.
class ContextFreeMLP(nn.Module):
    def __init__(self, input_dim=25, hidden=128, layers=3):
        super().__init__()
        mods, d = [], input_dim
        for _ in range(layers - 1):
            mods += [nn.Linear(d, hidden), nn.ReLU()]; d = hidden
        mods += [nn.Linear(d, 1)]
        self.f = nn.Sequential(*mods)
    def forward(self, xi, xs):
        return self.f(make_input(xi, xs)).squeeze(-1)

torch.manual_seed(0)
mlp = ContextFreeMLP().to(dev)
print("params:", sum(p.numel() for p in mlp.parameters()))
t0 = time.time()
mlp, ep = train_model(mlp, seed=0)
print(f"Context-free MLP seed 0: epochs per phase {ep} | {time.time() - t0:.0f} s")
for k in ["train", "val", "test"]:
    nll, acc = evaluate(mlp, k)
    print(f"  {k:5s} NLL {nll:.4f} | acc {acc:.4f}")
print(f"\nvalue of cross-mode context (seed 0): DeepHalo test 0.6880 − MLP test = {0.6880 - evaluate(mlp, 'test')[0]:+.4f} nats")

params: 19969
Context-free MLP seed 0: epochs per phase [18, 10] | 8 s
  train NLL 0.6235 | acc 0.7603
  val   NLL 0.6669 | acc 0.7437
  test  NLL 0.6829 | acc 0.7360

value of cross-mode context (seed 0): DeepHalo test 0.6880 − MLP test = +0.0051 nats


In [12]:
# Cell 10 — 5 seeds × {context-free MLP, DeepHalo} × {year split, random split}  (~8 min on a T4)
S_rand = {k: (torch.tensor(Xi_r[v]).to(dev), torch.tensor(Xs_r[v]).to(dev), torch.tensor(y[v]).to(dev))
          for k, v in idx.items()}
split_sets = {"year": S_year, "random": S_rand}
builders = {"MLP": lambda: ContextFreeMLP(),
            "DeepHalo": lambda: DeepHaloChoice(n=4, input_dim=25, H=8, L=4, embed=32)}

rows = []
for sname, Sd in split_sets.items():
    S = Sd
    for mname, build in builders.items():
        for seed in range(5):
            torch.manual_seed(seed)
            m, ep = train_model(build().to(dev), seed=seed)
            v, t = evaluate(m, "val")[0], evaluate(m, "test")[0]
            rows.append(dict(split=sname, model=mname, seed=seed, val=v, test=t))
            print(f"{sname:6s} {mname:8s} seed {seed} | epochs {ep} | val {v:.4f} | test {t:.4f}")
S = S_year   # back to the main protocol

res = pd.DataFrame(rows)
print("\nTest NLL, mean ± std over 5 seeds:")
print(res.groupby(["split", "model"])["test"].agg(["mean", "std"]).round(4))

print("\nDeepHalo − MLP (test NLL; NEGATIVE = cross-mode context helps):")
for sname in split_sets:
    a = res[(res.split == sname) & (res.model == "DeepHalo")]["test"]
    b = res[(res.split == sname) & (res.model == "MLP")]["test"]
    diff, se = a.mean() - b.mean(), np.sqrt(a.var() / len(a) + b.var() / len(b))
    print(f"  {sname:6s}: {diff:+.4f} ± {se:.4f} (s.e.)")
print("\npaper (random split): MLP 0.6870 ± 0.0049 | DeepHalo 0.6407 ± 0.0045")

year   MLP      seed 0 | epochs [18, 10] | val 0.6669 | test 0.6829
year   MLP      seed 1 | epochs [22, 10] | val 0.6644 | test 0.6843
year   MLP      seed 2 | epochs [14, 15] | val 0.6692 | test 0.6787
year   MLP      seed 3 | epochs [26, 11] | val 0.6624 | test 0.6830
year   MLP      seed 4 | epochs [18, 10] | val 0.6650 | test 0.6888
year   DeepHalo seed 0 | epochs [18, 10] | val 0.6699 | test 0.6880
year   DeepHalo seed 1 | epochs [17, 10] | val 0.6746 | test 0.6904
year   DeepHalo seed 2 | epochs [14, 12] | val 0.6639 | test 0.6848
year   DeepHalo seed 3 | epochs [20, 11] | val 0.6686 | test 0.6881
year   DeepHalo seed 4 | epochs [20, 11] | val 0.6670 | test 0.6891
random MLP      seed 0 | epochs [43, 13] | val 0.6384 | test 0.6387
random MLP      seed 1 | epochs [46, 17] | val 0.6338 | test 0.6382
random MLP      seed 2 | epochs [30, 16] | val 0.6392 | test 0.6375
random MLP      seed 3 | epochs [36, 17] | val 0.6383 | test 0.6378
random MLP      seed 4 | epochs [40, 15] | val 0

In [13]:
# Cell 11 — Thesis self-attention model (1 layer, width 32, 2 heads, no positional encoding, no dropout), 5 seeds × 2 splits
class SelfAttnChoice(nn.Module):
    def __init__(self, input_dim=25, width=32, heads=2, ffn=64, layers=1):
        super().__init__()
        self.embed = nn.Linear(input_dim, width); self.ln0 = nn.LayerNorm(width)
        self.blocks = nn.ModuleList([nn.ModuleDict(dict(
            attn=nn.MultiheadAttention(width, heads, batch_first=True),
            ln1=nn.LayerNorm(width),
            ffn=nn.Sequential(nn.Linear(width, ffn), nn.ReLU(), nn.Linear(ffn, width)),
            ln2=nn.LayerNorm(width))) for _ in range(layers)])
        self.out = nn.Linear(width, 1)
    def forward(self, xi, xs):
        h = self.ln0(self.embed(make_input(xi, xs)))
        for b in self.blocks:
            h = b["ln1"](h + b["attn"](h, h, h, need_weights=False)[0])
            h = b["ln2"](h + b["ffn"](h))
        return self.out(h).squeeze(-1)

print("SA params:", sum(p.numel() for p in SelfAttnChoice().parameters()))
for sname, Sd in split_sets.items():
    S = Sd
    for seed in range(5):
        torch.manual_seed(seed)
        m, ep = train_model(SelfAttnChoice().to(dev), seed=seed)
        v, t = evaluate(m, "val")[0], evaluate(m, "test")[0]
        rows.append(dict(split=sname, model="SelfAttn", seed=seed, val=v, test=t))
        print(f"{sname:6s} SelfAttn seed {seed} | epochs {ep} | val {v:.4f} | test {t:.4f}")
S = S_year

res = pd.DataFrame(rows)
print("\nTest NLL, mean ± std over 5 seeds:")
print(res.groupby(["split", "model"])["test"].agg(["mean", "std"]).round(4))
for sname in split_sets:
    a = res[(res.split == sname) & (res.model == "SelfAttn")]["test"]
    b = res[(res.split == sname) & (res.model == "MLP")]["test"]
    print(f"SelfAttn − MLP ({sname}): {a.mean() - b.mean():+.4f} ± {np.sqrt(a.var()/5 + b.var()/5):.4f}  (negative = context helps)")

SA params: 9473
year   SelfAttn seed 0 | epochs [18, 11] | val 0.6594 | test 0.6766
year   SelfAttn seed 1 | epochs [30, 12] | val 0.6629 | test 0.6821
year   SelfAttn seed 2 | epochs [15, 14] | val 0.6605 | test 0.6758
year   SelfAttn seed 3 | epochs [15, 22] | val 0.6638 | test 0.6784
year   SelfAttn seed 4 | epochs [16, 13] | val 0.6568 | test 0.6776
random SelfAttn seed 0 | epochs [28, 14] | val 0.6466 | test 0.6411
random SelfAttn seed 1 | epochs [39, 17] | val 0.6374 | test 0.6405
random SelfAttn seed 2 | epochs [35, 23] | val 0.6399 | test 0.6399
random SelfAttn seed 3 | epochs [36, 17] | val 0.6416 | test 0.6375
random SelfAttn seed 4 | epochs [40, 15] | val 0.6398 | test 0.6413

Test NLL, mean ± std over 5 seeds:
                   mean     std
split  model                   
random DeepHalo  0.6404  0.0016
       MLP       0.6382  0.0006
       SelfAttn  0.6401  0.0015
year   DeepHalo  0.6881  0.0021
       MLP       0.6835  0.0036
       SelfAttn  0.6781  0.0025
SelfAttn − M

In [14]:
# Cell 12 — Controls on the main (year, household-grouped) split, 5 seeds each:
#   (a) SelfAttn-selfonly: identical model, but each mode attends only to itself -> no cross-mode information
#   (b) MLP-85: context-free MLP size-matched to SelfAttn (~9.5k params)
class SelfAttnSelfOnly(SelfAttnChoice):
    def forward(self, xi, xs):
        h = self.ln0(self.embed(make_input(xi, xs)))
        mask = ~torch.eye(h.shape[1], dtype=torch.bool, device=h.device)   # True = blocked
        for b in self.blocks:
            h = b["ln1"](h + b["attn"](h, h, h, attn_mask=mask, need_weights=False)[0])
            h = b["ln2"](h + b["ffn"](h))
        return self.out(h).squeeze(-1)

controls = {"SelfAttn-selfonly": lambda: SelfAttnSelfOnly(),
            "MLP-85": lambda: ContextFreeMLP(hidden=85)}
for name, build in controls.items():
    print(f"{name} params:", sum(p.numel() for p in build().parameters()))

S = S_year
for name, build in controls.items():
    for seed in range(5):
        torch.manual_seed(seed)
        m, ep = train_model(build().to(dev), seed=seed)
        v, t = evaluate(m, "val")[0], evaluate(m, "test")[0]
        rows.append(dict(split="year", model=name, seed=seed, val=v, test=t))
        print(f"year {name:17s} seed {seed} | epochs {ep} | val {v:.4f} | test {t:.4f}")

res = pd.DataFrame(rows)
yr = res[res.split == "year"]
print("\nYear split (held-out households), mean ± std over 5 seeds:")
print(yr.groupby("model")[["val", "test"]].agg(["mean", "std"]).round(4))
ref = yr[yr.model == "SelfAttn"]["test"]
for name in ["SelfAttn-selfonly", "MLP-85", "MLP", "DeepHalo"]:
    b = yr[yr.model == name]["test"]
    print(f"SelfAttn − {name:17s}: {ref.mean() - b.mean():+.4f} ± {np.sqrt(ref.var()/5 + b.var()/5):.4f}")

SelfAttn-selfonly params: 9473
MLP-85 params: 9606
year SelfAttn-selfonly seed 0 | epochs [24, 20] | val 0.6593 | test 0.6792
year SelfAttn-selfonly seed 1 | epochs [32, 10] | val 0.6619 | test 0.6787
year SelfAttn-selfonly seed 2 | epochs [27, 10] | val 0.6583 | test 0.6878
year SelfAttn-selfonly seed 3 | epochs [26, 11] | val 0.6666 | test 0.6810
year SelfAttn-selfonly seed 4 | epochs [25, 10] | val 0.6597 | test 0.6831
year MLP-85            seed 0 | epochs [18, 10] | val 0.6664 | test 0.6821
year MLP-85            seed 1 | epochs [22, 10] | val 0.6643 | test 0.6829
year MLP-85            seed 2 | epochs [21, 10] | val 0.6648 | test 0.6821
year MLP-85            seed 3 | epochs [26, 10] | val 0.6611 | test 0.6818
year MLP-85            seed 4 | epochs [18, 10] | val 0.6625 | test 0.6821

Year split (held-out households), mean ± std over 5 seeds:
                      val            test        
                     mean     std    mean     std
model                                  

In [15]:
# Cell 13 — 10 more seeds (5–14) of SelfAttn and SelfAttn-selfonly on the year split → 15 seeds each
S = S_year
pair_models = {"SelfAttn": lambda: SelfAttnChoice(), "SelfAttn-selfonly": lambda: SelfAttnSelfOnly()}
for seed in range(5, 15):
    for name, build in pair_models.items():
        torch.manual_seed(seed)
        m, ep = train_model(build().to(dev), seed=seed)
        v, t = evaluate(m, "val")[0], evaluate(m, "test")[0]
        rows.append(dict(split="year", model=name, seed=seed, val=v, test=t))
        print(f"seed {seed:2d} {name:17s} | epochs {ep} | val {v:.4f} | test {t:.4f}")

res = pd.DataFrame(rows)
yr = res[(res.split == "year") & res.model.isin(pair_models)]
piv = yr.pivot_table(index="seed", columns="model", values=["val", "test"])
print(f"\nseeds per model: {len(piv)}")
for metric in ["val", "test"]:
    d = piv[(metric, "SelfAttn")] - piv[(metric, "SelfAttn-selfonly")]     # same seed = same init seed + batch order
    print(f"{metric:4s}: SelfAttn − selfonly = {d.mean():+.4f} ± {d.std(ddof=1)/np.sqrt(len(d)):.4f} (s.e.) | "
          f"SelfAttn better in {(d < 0).sum()}/{len(d)} seeds | median {d.median():+.4f}")

seed  5 SelfAttn          | epochs [24, 14] | val 0.6556 | test 0.6789
seed  5 SelfAttn-selfonly | epochs [24, 15] | val 0.6582 | test 0.6792
seed  6 SelfAttn          | epochs [26, 11] | val 0.6562 | test 0.6752
seed  6 SelfAttn-selfonly | epochs [26, 11] | val 0.6573 | test 0.6770
seed  7 SelfAttn          | epochs [24, 10] | val 0.6536 | test 0.6755
seed  7 SelfAttn-selfonly | epochs [19, 17] | val 0.6549 | test 0.6740
seed  8 SelfAttn          | epochs [17, 16] | val 0.6642 | test 0.6798
seed  8 SelfAttn-selfonly | epochs [34, 11] | val 0.6637 | test 0.6822
seed  9 SelfAttn          | epochs [26, 12] | val 0.6662 | test 0.6821
seed  9 SelfAttn-selfonly | epochs [33, 16] | val 0.6564 | test 0.6766
seed 10 SelfAttn          | epochs [26, 10] | val 0.6618 | test 0.6825
seed 10 SelfAttn-selfonly | epochs [26, 13] | val 0.6563 | test 0.6784
seed 11 SelfAttn          | epochs [27, 11] | val 0.6602 | test 0.6758
seed 11 SelfAttn-selfonly | epochs [20, 18] | val 0.6570 | test 0.6785
seed 1

In [16]:
# Cell 14 — Size-matched DeepHalo (embed 15 → ~10.2k used params vs SelfAttn 9.5k), 5 seeds on the year split
def used_params(m):
    unused = {"net.qualinear1.weight", "net.qualinear1.bias", "net.qualinear2.weight", "net.qualinear2.bias"}
    return sum(p.numel() for n, p in m.named_parameters() if n not in unused)

for e in [32, 15]:
    m = DeepHaloChoice(n=4, input_dim=25, H=8, L=4, embed=e)
    print(f"DeepHalo embed {e}: total {sum(p.numel() for p in m.parameters()):,} | used {used_params(m):,}")
print(f"SelfAttn: {sum(p.numel() for p in SelfAttnChoice().parameters()):,}\n")

S = S_year
for seed in range(5):
    torch.manual_seed(seed)
    m, ep = train_model(DeepHaloChoice(n=4, input_dim=25, H=8, L=4, embed=15).to(dev), seed=seed)
    v, t = evaluate(m, "val")[0], evaluate(m, "test")[0]
    rows.append(dict(split="year", model="DeepHalo-small", seed=seed, val=v, test=t))
    print(f"DeepHalo-small seed {seed} | epochs {ep} | val {v:.4f} | test {t:.4f}")

res = pd.DataFrame(rows)
yr = res[res.split == "year"]
order = ["SelfAttn", "SelfAttn-selfonly", "MLP-85", "MLP", "DeepHalo-small", "DeepHalo"]
summ = yr.groupby("model")["test"].agg(["mean", "std", "count"]).reindex(order).round(4)
print("\nYear split test NLL:\n", summ)
a = yr[yr.model == "SelfAttn"]["test"]; b = yr[yr.model == "DeepHalo-small"]["test"]
print(f"\nSelfAttn − DeepHalo-small: {a.mean() - b.mean():+.4f} ± {np.sqrt(a.var()/len(a) + b.var()/len(b)):.4f} (s.e.)")

DeepHalo embed 32: total 44,481 | used 42,369
DeepHalo embed 15: total 10,668 | used 10,188
SelfAttn: 9,473

DeepHalo-small seed 0 | epochs [18, 26] | val 0.6610 | test 0.6776
DeepHalo-small seed 1 | epochs [32, 10] | val 0.6627 | test 0.6834
DeepHalo-small seed 2 | epochs [27, 13] | val 0.6579 | test 0.6783
DeepHalo-small seed 3 | epochs [31, 10] | val 0.6580 | test 0.6820
DeepHalo-small seed 4 | epochs [25, 13] | val 0.6624 | test 0.6812

Year split test NLL:
                      mean     std  count
model                                   
SelfAttn           0.6790  0.0030     15
SelfAttn-selfonly  0.6797  0.0034     15
MLP-85             0.6822  0.0004      5
MLP                0.6835  0.0036      5
DeepHalo-small     0.6805  0.0025      5
DeepHalo           0.6881  0.0021      5

SelfAttn − DeepHalo-small: -0.0015 ± 0.0014 (s.e.)


In [17]:
# Expedia E1 — Load the ICDM-2013 train file, keep only randomized searches (random_bool = 1), count what we have
import glob
cands = [c for c in glob.glob("/kaggle/input/**/train.csv", recursive=True) if "expedia" in c.lower()]
print("train.csv found:", cands)
EXP_PATH = cands[0]

parts = []
for ch in pd.read_csv(EXP_PATH, chunksize=1_000_000):       # chunked: the file is ~2.3 GB
    parts.append(ch[ch["random_bool"] == 1])
ex = pd.concat(parts, ignore_index=True); del parts
print(f"randomized rows: {len(ex):,} | searches: {ex['srch_id'].nunique():,} | columns: {ex.shape[1]}")

per = ex.groupby("srch_id").agg(n=("prop_id", "size"), books=("booking_bool", "sum"), clicks=("click_bool", "sum"))
print("\nbookings per search:", per["books"].value_counts().sort_index().to_dict())
one = per[per["books"] == 1]
print(f"searches with exactly one booking: {len(one):,}")
print(f"slate size there: min {one.n.min()} | median {int(one.n.median())} | mean {one.n.mean():.1f} | max {one.n.max()}")
print("slate size histogram:", pd.cut(one.n, [0, 5, 10, 15, 20, 25, 30, 35, 40]).value_counts().sort_index().to_dict())

ex1 = ex[ex["srch_id"].isin(one.index)]
print("\nimpressions per position (single-booking searches):")
print(ex1["position"].value_counts().sort_index().to_dict())
print("booked hotel's position (top 10):", ex1.loc[ex1.booking_bool == 1, "position"].value_counts().head(10).to_dict())

train.csv found: ['/kaggle/input/datasets/ajazhussainda25m006/expedia-dataset/train.csv']
randomized rows: 2,939,652 | searches: 121,546 | columns: 54

bookings per search: {0: 105701, 1: 15845}
searches with exactly one booking: 15,845
slate size there: min 5 | median 27 | mean 23.8 | max 38
slate size histogram: {Interval(0, 5, closed='right'): 433, Interval(5, 10, closed='right'): 1844, Interval(10, 15, closed='right'): 1593, Interval(15, 20, closed='right'): 1566, Interval(20, 25, closed='right'): 1578, Interval(25, 30, closed='right'): 2672, Interval(30, 35, closed='right'): 6143, Interval(35, 40, closed='right'): 16}

impressions per position (single-booking searches):
{1: 15769, 2: 15751, 3: 15767, 4: 15276, 5: 773, 6: 15746, 7: 15329, 8: 14894, 9: 14564, 10: 13422, 11: 954, 12: 13827, 13: 13504, 14: 13204, 15: 12860, 16: 11661, 17: 1009, 18: 12217, 19: 11946, 20: 11638, 21: 11302, 22: 10162, 23: 971, 24: 10705, 25: 10402, 26: 10112, 27: 9811, 28: 9520, 29: 9206, 30: 8904, 31: 8

In [18]:
# Expedia E2 — Keep single-booking searches; build item features (vary across hotels) and shared features (constant per search)
ex = ex[ex["srch_id"].isin(one.index)].copy(); del ex1
ex = ex.sort_values(["srch_id", "position"]).reset_index(drop=True)

# ---- item features
ex["log_price"]          = np.log(ex["price_usd"].clip(10, 1000))          # clip, not drop: keeps choice sets intact
ex["review_missing"]     = ex["prop_review_score"].isna().astype(float)
ex["prop_review_score"]  = ex["prop_review_score"].fillna(0)
ex["loc2_missing"]       = ex["prop_location_score2"].isna().astype(float)
ex["prop_location_score2"] = ex["prop_location_score2"].fillna(0)
ex["dist_missing"]       = ex["orig_destination_distance"].isna().astype(float)
ex["log_dist"]           = np.log1p(ex["orig_destination_distance"].fillna(0))
ex["hist_price_missing"] = (ex["prop_log_historical_price"] == 0).astype(float)   # 0 = not sold last period
rate_cols = [f"comp{i}_rate" for i in range(1, 9)]
inv_cols  = [f"comp{i}_inv"  for i in range(1, 9)]
ex["comp_cheaper"] = (ex[rate_cols] == 1).sum(axis=1)    # competitors Expedia is cheaper than
ex["comp_dearer"]  = (ex[rate_cols] == -1).sum(axis=1)   # competitors Expedia is dearer than
ex["comp_noavail"] = (ex[inv_cols] == 1).sum(axis=1)     # competitors with no availability

EX_ITEM = ["prop_starrating", "prop_review_score", "review_missing", "prop_brand_bool",
           "prop_location_score1", "prop_location_score2", "loc2_missing",
           "prop_log_historical_price", "hist_price_missing", "log_price", "promotion_flag",
           "log_dist", "dist_missing", "comp_cheaper", "comp_dearer", "comp_noavail"]

# ---- shared (search-level) features
ex["log_booking_window"]  = np.log1p(ex["srch_booking_window"])
ex["vis_star_missing"]    = ex["visitor_hist_starrating"].isna().astype(float)
ex["vis_star"]            = ex["visitor_hist_starrating"].fillna(0)
ex["vis_adr_missing"]     = ex["visitor_hist_adr_usd"].isna().astype(float)
ex["log_vis_adr"]         = np.log1p(ex["visitor_hist_adr_usd"].fillna(0))
EX_SHARED = ["srch_length_of_stay", "log_booking_window", "srch_adults_count", "srch_children_count",
             "srch_room_count", "srch_saturday_night_bool", "vis_star", "vis_star_missing",
             "log_vis_adr", "vis_adr_missing"]

print(f"rows {len(ex):,} | searches {ex.srch_id.nunique():,} | item feats {len(EX_ITEM)} | shared feats {len(EX_SHARED)}")
print("NaNs left in features:", int(ex[EX_ITEM + EX_SHARED].isna().sum().sum()))
const = ex.groupby("srch_id")[EX_SHARED].nunique().max()
print("shared features truly constant within a search:", bool((const == 1).all()))
print("\nitem features (booked vs not booked, mean):")
print(ex.groupby("booking_bool")[EX_ITEM].mean().T.round(3))
print("\ndate range:", ex["date_time"].min(), "→", ex["date_time"].max())

rows 376,960 | searches 15,845 | item feats 16 | shared feats 10
NaNs left in features: 0
shared features truly constant within a search: True

item features (booked vs not booked, mean):
booking_bool                   0      1
prop_starrating            3.066  3.251
prop_review_score          3.713  3.903
review_missing             0.002  0.001
prop_brand_bool            0.664  0.683
prop_location_score1       2.664  2.720
prop_location_score2       0.096  0.179
loc2_missing               0.271  0.113
prop_log_historical_price  4.248  4.285
hist_price_missing         0.146  0.145
log_price                  4.801  4.742
promotion_flag             0.164  0.256
log_dist                   4.077  4.021
dist_missing               0.309  0.320
comp_cheaper               0.221  0.284
comp_dearer                0.213  0.175
comp_noavail               0.109  0.108

date range: 2012-11-01 01:09:50 → 2013-06-30 23:34:32


In [24]:
# Expedia E2b — The test collapse is a data artifact: prop_log_historical_price becomes 0 ("unknown") in the final weeks
wk = pd.to_datetime(ex["date_time"]).dt.to_period("W")
frac0 = (ex["prop_log_historical_price"] == 0).groupby(wk).mean() * 100
print("% hotels with historical price = 0 — first 5 weeks:", frac0.head(5).round(1).to_dict())
print("% hotels with historical price = 0 — last 8 weeks :", frac0.tail(8).round(1).to_dict())

EX_ITEM = [c for c in EX_ITEM if c not in ("prop_log_historical_price", "hist_price_missing")]
D_I, D_S = len(EX_ITEM), len(EX_SHARED)
print(f"\nitem features now: {D_I} | shared: {D_S}")

% hotels with historical price = 0 — first 5 weeks: {Period('2012-10-29/2012-11-04', 'W-SUN'): 0.3, Period('2012-11-05/2012-11-11', 'W-SUN'): 0.5, Period('2012-11-12/2012-11-18', 'W-SUN'): 0.3, Period('2012-11-19/2012-11-25', 'W-SUN'): 0.6, Period('2012-11-26/2012-12-02', 'W-SUN'): 0.3}
% hotels with historical price = 0 — last 8 weeks : {Period('2013-05-06/2013-05-12', 'W-SUN'): 0.7, Period('2013-05-13/2013-05-19', 'W-SUN'): 0.4, Period('2013-05-20/2013-05-26', 'W-SUN'): 0.5, Period('2013-05-27/2013-06-02', 'W-SUN'): 0.6, Period('2013-06-03/2013-06-09', 'W-SUN'): 100.0, Period('2013-06-10/2013-06-16', 'W-SUN'): 100.0, Period('2013-06-17/2013-06-23', 'W-SUN'): 100.0, Period('2013-06-24/2013-06-30', 'W-SUN'): 100.0}

item features now: 14 | shared: 10


In [25]:
# Expedia E3 — Padded slate tensors, split (time-based by default), scaling on train only
SPLIT = "time"          # "time": latest 20% of searches = test | "random": random 20% = test
MAXN = int(ex.groupby("srch_id").size().max())
sidx, srch_ids = pd.factorize(ex["srch_id"], sort=True)
slot = ex.groupby("srch_id").cumcount().values
nS = len(srch_ids)

ex_Xi   = np.zeros((nS, MAXN, len(EX_ITEM)), np.float32); ex_Xi[sidx, slot] = ex[EX_ITEM].values
ex_pos  = np.zeros((nS, MAXN), np.int64);                 ex_pos[sidx, slot] = ex["position"].values   # 0 = padding
ex_mask = np.zeros((nS, MAXN), bool);                     ex_mask[sidx, slot] = True
ex_y    = np.zeros(nS, np.int64)
booked  = ex["booking_bool"].values == 1
ex_y[sidx[booked]] = slot[booked]
first   = ex.groupby("srch_id", sort=True).first()
ex_Xs   = first[EX_SHARED].values.astype(np.float32)
ex_date = pd.to_datetime(first["date_time"]).values

rng = np.random.default_rng(0)
if SPLIT == "time":
    is_te = ex_date >= np.quantile(ex_date.astype("int64"), 0.8).astype("datetime64[ns]")
else:
    is_te = rng.random(nS) < 0.2
rest = np.where(~is_te)[0]
is_va = np.zeros(nS, bool); is_va[rng.choice(rest, size=int(0.1 * nS), replace=False)] = True
is_tr = ~is_te & ~is_va

# scale continuous features with TRAIN statistics only (padding stays 0)
item_bin   = {"review_missing", "prop_brand_bool", "loc2_missing", "hist_price_missing", "promotion_flag", "dist_missing"}
shared_bin = {"srch_saturday_night_bool", "vis_star_missing", "vis_adr_missing"}
ci = [j for j, c in enumerate(EX_ITEM) if c not in item_bin]
cs = [j for j, c in enumerate(EX_SHARED) if c not in shared_bin]
real_tr = ex_Xi[is_tr][ex_mask[is_tr]]
mu_i, sd_i = real_tr[:, ci].mean(0), real_tr[:, ci].std(0)
ex_Xi[:, :, ci] = (ex_Xi[:, :, ci] - mu_i) / sd_i
ex_Xi[~ex_mask] = 0.0
mu_s, sd_s = ex_Xs[is_tr][:, cs].mean(0), ex_Xs[is_tr][:, cs].std(0)
ex_Xs[:, cs] = (ex_Xs[:, cs] - mu_s) / sd_s

ex_masks = {"train": is_tr, "val": is_va, "test": is_te}
ex_S = {k: tuple(torch.tensor(a[m]).to(dev) for a in (ex_Xi, ex_Xs, ex_pos, ex_mask, ex_y))
        for k, m in ex_masks.items()}

nslate = ex_mask.sum(1)
for k, m in ex_masks.items():
    d = pd.to_datetime(ex_date[m])
    print(f"{k:5s}: {m.sum():6,} searches | {d.min().date()} → {d.max().date()} | mean slate {nslate[m].mean():.1f} | "
          f"booked at pos 1: {(ex_pos[m, ex_y[m]] == 1).mean()*100:.1f}% | uniform-guess NLL {np.log(nslate[m]).mean():.4f}")
print("\nshapes:", {k: tuple(t.shape) for k, t in zip(["x_item", "x_shared", "pos", "mask", "y"], ex_S["train"])})

train: 11,092 searches | 2012-11-01 → 2013-05-22 | mean slate 23.9 | booked at pos 1: 13.7% | uniform-guess NLL 3.0572
val  :  1,584 searches | 2012-11-01 → 2013-05-22 | mean slate 24.1 | booked at pos 1: 13.7% | uniform-guess NLL 3.0618
test :  3,169 searches | 2013-05-22 → 2013-06-30 | mean slate 23.3 | booked at pos 1: 15.5% | uniform-guess NLL 3.0282

shapes: {'x_item': (11092, 38, 14), 'x_shared': (11092, 10), 'pos': (11092, 38), 'mask': (11092, 38), 'y': (11092,)}


In [26]:
# Expedia E4 (v2) — Masked models for variable-size slates: logit_i = core(slate)_i + position_bias(pos_i)
# Input sizes now follow the feature lists (E2b dropped the two historical-price features)
D_I, D_S = len(EX_ITEM), len(EX_SHARED)
assert ex_S["train"][0].shape[-1] == D_I, "rerun E3 after E2b first"
print(f"item features: {D_I} | shared features: {D_S} | per-hotel input: {D_I + D_S}")
NEG = -1e9

class PosBias(nn.Module):                              # one learned scalar per display position (0 = padding)
    def __init__(self, n_pos=41):
        super().__init__(); self.b = nn.Embedding(n_pos, 1); nn.init.zeros_(self.b.weight)
    def forward(self, pos):
        return self.b(pos).squeeze(-1)

def ex_input(xi, xs):                                  # each hotel sees [its item feats | the search feats]
    return torch.cat([xi, xs.unsqueeze(1).expand(-1, xi.shape[1], -1)], dim=-1)

class ExModel(nn.Module):
    def __init__(self, core):
        super().__init__(); self.core = core; self.pos = PosBias()
    def forward(self, xi, xs, pos, mask):
        return (self.core(xi, xs, mask) + self.pos(pos)).masked_fill(~mask, NEG)

class ExMNL_A(nn.Module):                              # linear in item features
    def __init__(self, d_i=D_I):
        super().__init__(); self.beta = nn.Linear(d_i, 1, bias=False)
    def forward(self, xi, xs, mask):
        return self.beta(xi).squeeze(-1)

class ExMNL_B(ExMNL_A):                                # + item × search interactions
    def __init__(self, d_i=D_I, d_s=D_S):
        super().__init__(d_i); self.G = nn.Parameter(torch.zeros(d_i, d_s))
    def forward(self, xi, xs, mask):
        return super().forward(xi, xs, mask) + torch.einsum("bnd,de,be->bn", xi, self.G, xs)

class ExMLP(nn.Module):                                # context-free: own features + search only
    def __init__(self, d_in=D_I + D_S, hidden=85):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, xi, xs, mask):
        return self.f(ex_input(xi, xs)).squeeze(-1)

class ExDeepHalo(nn.Module):                           # authors' DeepHalo layers; padded hotels zeroed before every aggregation
    def __init__(self, d_in=D_I + D_S, H=8, L=4, embed=15):
        super().__init__(); self.net = DeepHalo(n=MAXN, input_dim=d_in, H=H, L=L, embed=embed)
    def forward(self, xi, xs, mask):
        net, m = self.net, mask.unsqueeze(-1).float()
        lengths = mask.sum(1).float()
        Z = net.enc_norm(net.basic_encoder(ex_input(xi, xs))) * m
        X = Z.clone()
        for fc, nt in zip(net.aggregate_linear, net.nonlinear):
            Z_bar = ((fc(Z) * m).sum(1) / lengths.unsqueeze(1)).unsqueeze(-1).unsqueeze(1)
            phi = nt(X) * m.unsqueeze(-1)
            Z = ((phi * Z_bar).sum(2) / net.H + Z) * m
        return net.final_linear(Z).squeeze(-1)

class ExSelfAttn(nn.Module):                           # thesis model: 1 layer, width 32, 2 heads, no positional encoding
    def __init__(self, d_in=D_I + D_S, width=32, heads=2, ffn=64, self_only=False):
        super().__init__()
        self.self_only = self_only
        self.embed = nn.Linear(d_in, width); self.ln0 = nn.LayerNorm(width)
        self.attn = nn.MultiheadAttention(width, heads, batch_first=True); self.ln1 = nn.LayerNorm(width)
        self.ffn = nn.Sequential(nn.Linear(width, ffn), nn.ReLU(), nn.Linear(ffn, width)); self.ln2 = nn.LayerNorm(width)
        self.out = nn.Linear(width, 1)
    def forward(self, xi, xs, mask):
        h = self.ln0(self.embed(ex_input(xi, xs)))
        if self.self_only:                             # each hotel attends only to itself -> no cross-hotel information
            blocked = ~torch.eye(h.shape[1], dtype=torch.bool, device=h.device)
            a = self.attn(h, h, h, attn_mask=blocked, need_weights=False)[0]
        else:
            a = self.attn(h, h, h, key_padding_mask=~mask, need_weights=False)[0]
        h = self.ln1(h + a)
        h = self.ln2(h + self.ffn(h))
        return self.out(h).squeeze(-1)

def ex_evaluate(model, split, bs=1024):
    xi, xs, pos, mk, yy = ex_S[split]; model.eval(); nll, top1 = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(yy), bs):
            lg = model(xi[i:i+bs], xs[i:i+bs], pos[i:i+bs], mk[i:i+bs])
            nll += F.cross_entropy(lg, yy[i:i+bs], reduction="sum").item()
            top1 += (lg.argmax(1) == yy[i:i+bs]).sum().item()
    return nll / len(yy), top1 / len(yy)

def ex_train(model, phases=((1e-3, 10), (1e-4, 10)), bs=256, max_epochs=300, seed=0):
    g = torch.Generator().manual_seed(seed)
    xi, xs, pos, mk, yy = ex_S["train"]; n = len(yy)
    best_v, best_state = ex_evaluate(model, "val")[0], copy.deepcopy(model.state_dict()); epochs_run = []
    for lr, patience in phases:
        model.load_state_dict(best_state)
        opt = torch.optim.Adam(model.parameters(), lr=lr); wait = 0
        for ep in range(max_epochs):
            model.train()
            perm = torch.randperm(n, generator=g).to(dev)
            for i in range(0, n, bs):
                b = perm[i:i+bs]
                loss = F.cross_entropy(model(xi[b], xs[b], pos[b], mk[b]), yy[b])
                opt.zero_grad(); loss.backward(); opt.step()
            v = ex_evaluate(model, "val")[0]
            if v < best_v - 1e-5: best_v, best_state, wait = v, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
                if wait >= patience: break
        epochs_run.append(ep + 1)
    model.load_state_dict(best_state)
    return model, epochs_run

EX_MODELS = {
    "MNL-A":             lambda: ExModel(ExMNL_A()),
    "MNL-B":             lambda: ExModel(ExMNL_B()),
    "MLP-85":            lambda: ExModel(ExMLP(hidden=85)),
    "DeepHalo-15":       lambda: ExModel(ExDeepHalo(embed=15)),
    "SelfAttn":          lambda: ExModel(ExSelfAttn()),
    "SelfAttn-selfonly": lambda: ExModel(ExSelfAttn(self_only=True)),
}

# ---- smoke tests: sizes, no NaNs, padding never chosen, permutation equivariance (shuffle hotels+positions together)
xi, xs, pos, mk, yy = (t[:64] for t in ex_S["train"])
perm = torch.randperm(MAXN, device=dev)
for name, build in EX_MODELS.items():
    torch.manual_seed(0); m = build().to(dev).eval()
    unused = sum(p.numel() for n_, p in m.named_parameters() if "qualinear" in n_)
    with torch.no_grad():
        lg = m(xi, xs, pos, mk)
        lg_p = m(xi[:, perm], xs, pos[:, perm], mk[:, perm])
        eq_err = (lg_p - lg[:, perm])[mk[:, perm]].abs().max().item()
    print(f"{name:17s} params {sum(p.numel() for p in m.parameters()) - unused:6,} | NaN {torch.isnan(lg).any().item()} | "
          f"max padded logit {lg[~mk].max().item():.0e} | equivariance error {eq_err:.1e}")

item features: 14 | shared features: 10 | per-hotel input: 24
MNL-A             params     55 | NaN False | max padded logit -1e+09 | equivariance error 0.0e+00
MNL-B             params    195 | NaN False | max padded logit -1e+09 | equivariance error 0.0e+00
MLP-85            params  9,562 | NaN False | max padded logit -1e+09 | equivariance error 0.0e+00
DeepHalo-15       params 10,214 | NaN False | max padded logit -1e+09 | equivariance error 2.4e-07
SelfAttn          params  9,482 | NaN False | max padded logit -1e+09 | equivariance error 3.0e-07
SelfAttn-selfonly params  9,482 | NaN False | max padded logit -1e+09 | equivariance error 0.0e+00


In [27]:
# Expedia E5 — First full comparison on the time split: convex/simple models 1 seed, neural models 5 seeds
class ExZero(nn.Module):                               # no content at all: score = position bias only
    def forward(self, xi, xs, mask):
        return torch.zeros(mask.shape, device=mask.device)
EX_MODELS["Position-only"] = lambda: ExModel(ExZero())

order = ["Position-only", "MNL-A", "MNL-B", "MLP-85", "DeepHalo-15", "SelfAttn", "SelfAttn-selfonly"]
n_seeds = {"Position-only": 1, "MNL-A": 1, "MNL-B": 1}
ex_rows = []
for name in order:
    for seed in range(n_seeds.get(name, 5)):
        torch.manual_seed(seed); t0 = time.time()
        m, ep = ex_train(EX_MODELS[name]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=name, seed=seed, val=v, test=t, acc=acc))
        print(f"{name:17s} seed {seed} | epochs {ep} | {time.time()-t0:4.0f}s | val {v:.4f} | test {t:.4f} | top-1 {acc:.4f}")

exr = pd.DataFrame(ex_rows)
print("\nTest NLL / top-1 (uniform guess on test = 3.0282):")
print(exr.groupby("model")[["val", "test", "acc"]].agg(["mean", "std"]).reindex(order).round(4))

def ex_gap(a, b):
    x, y = exr[exr.model == a]["test"], exr[exr.model == b]["test"]
    return f"{a} − {b}: {x.mean() - y.mean():+.4f} ± {np.sqrt(x.var()/len(x) + y.var()/len(y)):.4f}"
print("\nGaps (negative = first model better):")
for a, b in [("SelfAttn", "SelfAttn-selfonly"), ("DeepHalo-15", "MLP-85"), ("SelfAttn", "MLP-85"), ("SelfAttn", "DeepHalo-15")]:
    print(" ", ex_gap(a, b))

Position-only     seed 0 | epochs [209, 10] |   11s | val 2.8814 | test 2.8373 | top-1 0.1562
MNL-A             seed 0 | epochs [107, 10] |    7s | val 2.6529 | test 2.6343 | top-1 0.2149
MNL-B             seed 0 | epochs [69, 10] |    7s | val 2.6527 | test 2.6315 | top-1 0.2187
MLP-85            seed 0 | epochs [30, 11] |    4s | val 2.6054 | test 2.5716 | top-1 0.2275
MLP-85            seed 1 | epochs [27, 10] |    3s | val 2.6062 | test 2.5784 | top-1 0.2244
MLP-85            seed 2 | epochs [38, 10] |    4s | val 2.6044 | test 2.5883 | top-1 0.2256
MLP-85            seed 3 | epochs [32, 10] |    4s | val 2.6009 | test 2.5793 | top-1 0.2184
MLP-85            seed 4 | epochs [30, 10] |    4s | val 2.6048 | test 2.5777 | top-1 0.2222
DeepHalo-15       seed 0 | epochs [34, 10] |   46s | val 2.6153 | test 2.5703 | top-1 0.2294
DeepHalo-15       seed 1 | epochs [37, 10] |   49s | val 2.6122 | test 2.5741 | top-1 0.2300
DeepHalo-15       seed 2 | epochs [32, 10] |   44s | val 2.6133 | te

In [28]:
# Expedia E6 — Why do neural models collapse on test but not val?  (1) which features shifted, (2) is the loss driven by a few searches?
tr_it = ex_Xi[is_tr][ex_mask[is_tr]]; te_it = ex_Xi[is_te][ex_mask[is_te]]
tr_sh, te_sh = ex_Xs[is_tr], ex_Xs[is_te]
rows_ = []
for kind, names, A, B in [("item", EX_ITEM, tr_it, te_it), ("shared", EX_SHARED, tr_sh, te_sh)]:
    for j, c in enumerate(names):
        lo, hi = A[:, j].min(), A[:, j].max()
        rows_.append(dict(feature=c, kind=kind, train_mean=A[:, j].mean(), test_mean=B[:, j].mean(),
                          shift_sd=(B[:, j].mean() - A[:, j].mean()) / (A[:, j].std() + 1e-9),
                          pct_test_outside_train=100 * ((B[:, j] < lo) | (B[:, j] > hi)).mean()))
shift = pd.DataFrame(rows_).set_index("feature").round(3)
print("Feature shift, train → test (sorted by size, in train-std units):")
print(shift.reindex(shift.shift_sd.abs().sort_values(ascending=False).index))

def per_search_nll(model, split):
    xi, xs, pos, mk, yy = ex_S[split]; model.eval()
    with torch.no_grad():
        return F.cross_entropy(model(xi, xs, pos, mk), yy, reduction="none").cpu().numpy()

print("\nPer-search loss (seed 0 of each):")
for name in ["MNL-A", "MLP-85", "SelfAttn"]:
    torch.manual_seed(0); m, _ = ex_train(EX_MODELS[name]().to(dev), seed=0)
    for split in ["val", "test"]:
        l = per_search_nll(m, split); k = max(1, int(0.01 * len(l)))
        print(f"  {name:9s} {split:4s} | mean {l.mean():.3f} | median {np.median(l):.3f} | max {l.max():7.1f} | "
              f"worst 1% of searches = {np.sort(l)[::-1][:k].sum() / l.sum() * 100:4.1f}% of total loss")

Feature shift, train → test (sorted by size, in train-std units):
                            kind  train_mean  test_mean  shift_sd  \
feature                                                             
review_missing              item       0.001      0.006     0.223   
srch_children_count       shared       0.000      0.150     0.150   
log_price                   item       0.000      0.115     0.115   
log_vis_adr               shared      -0.000      0.056     0.056   
vis_star                  shared       0.000      0.055     0.055   
vis_star_missing          shared       0.939      0.927    -0.049   
vis_adr_missing           shared       0.939      0.927    -0.048   
prop_location_score2        item      -0.000     -0.043    -0.043   
prop_review_score           item      -0.000     -0.040    -0.040   
dist_missing                item       0.305      0.324     0.040   
srch_saturday_night_bool  shared       0.524      0.504    -0.040   
srch_room_count           shared     

In [29]:
# Expedia E7 — Firm up: (a) 10 more seeds SelfAttn vs selfonly, (b) DeepHalo at the paper's Expedia size (L=5, embed 32),
#              (c) LCL (Tomlinson & Benson): u_i = θ'x_i + x_i' A ȳ_S — the classic first-order context model
class ExLCL(nn.Module):
    def __init__(self, d_i=D_I):
        super().__init__()
        self.theta = nn.Linear(d_i, 1, bias=False)
        self.A = nn.Parameter(torch.zeros(d_i, d_i))
    def forward(self, xi, xs, mask):
        m = mask.unsqueeze(-1).float()
        ybar = (xi * m).sum(1) / m.sum(1)                      # slate mean of item features (includes the item itself)
        return self.theta(xi).squeeze(-1) + torch.einsum("bnd,de,be->bn", xi, self.A, ybar)

EX_MODELS["LCL"] = lambda: ExModel(ExLCL())
EX_MODELS["DeepHalo-paper"] = lambda: ExModel(ExDeepHalo(embed=32, L=5))
_m = EX_MODELS["DeepHalo-paper"]()
print("DeepHalo-paper params:", sum(p.numel() for n_, p in _m.named_parameters() if "qualinear" not in n_))

plan = [("LCL", range(1)), ("DeepHalo-paper", range(5)),
        ("SelfAttn", range(5, 15)), ("SelfAttn-selfonly", range(5, 15))]
for name, seeds in plan:
    for seed in seeds:
        torch.manual_seed(seed); t0 = time.time()
        m, ep = ex_train(EX_MODELS[name]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=name, seed=seed, val=v, test=t, acc=acc))
        print(f"{name:17s} seed {seed:2d} | epochs {ep} | {time.time()-t0:4.0f}s | val {v:.4f} | test {t:.4f}")

exr = pd.DataFrame(ex_rows)
order = ["Position-only", "MNL-A", "MNL-B", "LCL", "MLP-85", "DeepHalo-15", "DeepHalo-paper", "SelfAttn-selfonly", "SelfAttn"]
print("\nTest NLL:")
print(exr.groupby("model")["test"].agg(["mean", "std", "count"]).reindex(order).round(4))

pv = exr[exr.model.isin(["SelfAttn", "SelfAttn-selfonly"])].pivot_table(index="seed", columns="model", values="test")
d = pv["SelfAttn"] - pv["SelfAttn-selfonly"]
print(f"\nSelfAttn − selfonly over {len(d)} paired seeds: {d.mean():+.4f} ± {d.std(ddof=1)/np.sqrt(len(d)):.4f} | "
      f"SelfAttn better in {(d < 0).sum()}/{len(d)}")
for a, b in [("LCL", "MNL-A"), ("SelfAttn", "LCL"), ("DeepHalo-paper", "MLP-85"), ("SelfAttn", "DeepHalo-paper")]:
    print(" ", ex_gap(a, b))

DeepHalo-paper params: 52210
LCL               seed  0 | epochs [107, 10] |   11s | val 2.6155 | test 2.5965
DeepHalo-paper    seed  0 | epochs [20, 10] |   23s | val 2.6109 | test 2.5954
DeepHalo-paper    seed  1 | epochs [21, 10] |   24s | val 2.6217 | test 2.6013
DeepHalo-paper    seed  2 | epochs [16, 11] |   21s | val 2.6404 | test 2.6190
DeepHalo-paper    seed  3 | epochs [20, 11] |   24s | val 2.6263 | test 2.6023
DeepHalo-paper    seed  4 | epochs [20, 10] |   23s | val 2.6290 | test 2.6163
SelfAttn          seed  5 | epochs [33, 11] |    7s | val 2.5823 | test 2.5477
SelfAttn          seed  6 | epochs [32, 10] |    7s | val 2.5794 | test 2.5495
SelfAttn          seed  7 | epochs [28, 10] |    7s | val 2.5839 | test 2.5570
SelfAttn          seed  8 | epochs [31, 10] |    7s | val 2.5764 | test 2.5621
SelfAttn          seed  9 | epochs [33, 11] |    8s | val 2.5842 | test 2.5579
SelfAttn          seed 10 | epochs [26, 10] |    6s | val 2.5813 | test 2.5569
SelfAttn          seed

In [30]:
# Expedia E8 — What kind of context? Pooling ladder (others' mean / mean+sd), plus LCL with the item excluded from its own mean
def others_stats(xi, mask):                                    # mean and sd of the OTHER hotels' item features
    m = mask.unsqueeze(-1).float(); n = m.sum(1, keepdim=True)
    s1 = (xi * m).sum(1, keepdim=True); s2 = (xi**2 * m).sum(1, keepdim=True)
    mean_o = (s1 - xi) / (n - 1)
    var_o = ((s2 - xi**2) / (n - 1) - mean_o**2).clamp_min(0)
    return mean_o, var_o.sqrt()

class ExPool(nn.Module):
    def __init__(self, use_sd=False, hidden=79):
        super().__init__(); self.use_sd = use_sd
        d_in = D_I + D_S + D_I * (2 if use_sd else 1)
        self.f = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, xi, xs, mask):
        mo, so = others_stats(xi, mask)
        parts = [ex_input(xi, xs), mo] + ([so] if self.use_sd else [])
        return self.f(torch.cat(parts, -1)).squeeze(-1)

class ExLCLx(ExLCL):                                           # LCL with ȳ = mean of the OTHER hotels (no own quadratic term)
    def forward(self, xi, xs, mask):
        mo, _ = others_stats(xi, mask)
        return self.theta(xi).squeeze(-1) + torch.einsum("bnd,de,bnd->bn", xi, self.A, mo)

EX_MODELS["LCL-excl"]     = lambda: ExModel(ExLCLx())
EX_MODELS["Pool-mean"]    = lambda: ExModel(ExPool(use_sd=False, hidden=79))
EX_MODELS["Pool-mean+sd"] = lambda: ExModel(ExPool(use_sd=True,  hidden=74))
for k in ["Pool-mean", "Pool-mean+sd"]:
    print(k, "params:", sum(p.numel() for p in EX_MODELS[k]().parameters()))

for name, seeds in [("LCL-excl", range(1)), ("Pool-mean", range(15)), ("Pool-mean+sd", range(15))]:
    for seed in seeds:
        torch.manual_seed(seed)
        m, ep = ex_train(EX_MODELS[name]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=name, seed=seed, val=v, test=t, acc=acc))
        print(f"{name:13s} seed {seed:2d} | epochs {ep} | val {v:.4f} | test {t:.4f}")

exr = pd.DataFrame(ex_rows)
ladder = ["MNL-A", "LCL", "LCL-excl", "SelfAttn-selfonly", "Pool-mean", "Pool-mean+sd", "SelfAttn"]
print("\nTest NLL ladder:")
print(exr.groupby("model")["test"].agg(["mean", "std", "count"]).reindex(ladder).round(4))
for a, b in [("Pool-mean", "SelfAttn-selfonly"), ("Pool-mean+sd", "Pool-mean"), ("SelfAttn", "Pool-mean+sd"), ("LCL-excl", "MNL-A")]:
    print(" ", ex_gap(a, b))


Pool-mean params: 9522
Pool-mean+sd params: 9588
LCL-excl      seed  0 | epochs [60, 10] | val 2.6420 | test 2.6187
Pool-mean     seed  0 | epochs [25, 10] | val 2.6005 | test 2.5706
Pool-mean     seed  1 | epochs [28, 13] | val 2.5993 | test 2.5727
Pool-mean     seed  2 | epochs [28, 11] | val 2.6122 | test 2.5612
Pool-mean     seed  3 | epochs [32, 10] | val 2.6085 | test 2.5804
Pool-mean     seed  4 | epochs [24, 10] | val 2.6105 | test 2.5700
Pool-mean     seed  5 | epochs [31, 10] | val 2.6043 | test 2.5703
Pool-mean     seed  6 | epochs [27, 11] | val 2.6068 | test 2.5723
Pool-mean     seed  7 | epochs [31, 10] | val 2.6017 | test 2.5755
Pool-mean     seed  8 | epochs [24, 10] | val 2.6126 | test 2.5747
Pool-mean     seed  9 | epochs [28, 10] | val 2.6120 | test 2.5712
Pool-mean     seed 10 | epochs [30, 10] | val 2.6054 | test 2.5732
Pool-mean     seed 11 | epochs [27, 10] | val 2.6158 | test 2.5698
Pool-mean     seed 12 | epochs [30, 11] | val 2.6050 | test 2.5761
Pool-mean    

In [31]:
# Expedia E9 — Size fairness, both directions:
#   (a) DeepHalo-15 to 15 seeds (same as SelfAttn)
#   (b) SelfAttn and its self-only copy scaled UP to DeepHalo-paper size (~52k): does SA also overfit when big?
EX_MODELS["SelfAttn-w80"]          = lambda: ExModel(ExSelfAttn(width=80, ffn=160))
EX_MODELS["SelfAttn-w80-selfonly"] = lambda: ExModel(ExSelfAttn(width=80, ffn=160, self_only=True))
for k in ["DeepHalo-15", "DeepHalo-paper", "SelfAttn", "SelfAttn-w80"]:
    _m = EX_MODELS[k]()
    print(f"{k:15s} params {sum(p.numel() for n_, p in _m.named_parameters() if 'qualinear' not in n_):,}")

plan = [("DeepHalo-15", range(5, 15)), ("SelfAttn-w80", range(15)), ("SelfAttn-w80-selfonly", range(15))]
for name, seeds in plan:
    for seed in seeds:
        torch.manual_seed(seed); t0 = time.time()
        m, ep = ex_train(EX_MODELS[name]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=name, seed=seed, val=v, test=t, acc=acc))
        print(f"{name:21s} seed {seed:2d} | epochs {ep} | {time.time()-t0:4.0f}s | val {v:.4f} | test {t:.4f}")

exr = pd.DataFrame(ex_rows)
cmp_ = ["MLP-85", "DeepHalo-15", "SelfAttn-selfonly", "SelfAttn", "DeepHalo-paper", "SelfAttn-w80-selfonly", "SelfAttn-w80"]
print("\nTest NLL:")
print(exr.groupby("model")["test"].agg(["mean", "std", "count"]).reindex(cmp_).round(4))
print("\n~10k params:"); print(" ", ex_gap("SelfAttn", "DeepHalo-15"))
print("~52k params:"); print(" ", ex_gap("SelfAttn-w80", "DeepHalo-paper"))
print("context gain at ~52k:"); print(" ", ex_gap("SelfAttn-w80", "SelfAttn-w80-selfonly"))

DeepHalo-15     params 10,214
DeepHalo-paper  params 52,210
SelfAttn        params 9,482
SelfAttn-w80    params 54,362
DeepHalo-15           seed  5 | epochs [37, 10] |   50s | val 2.6175 | test 2.5859
DeepHalo-15           seed  6 | epochs [43, 10] |   55s | val 2.6072 | test 2.5787
DeepHalo-15           seed  7 | epochs [34, 10] |   46s | val 2.6119 | test 2.5840
DeepHalo-15           seed  8 | epochs [37, 11] |   50s | val 2.6128 | test 2.5838
DeepHalo-15           seed  9 | epochs [31, 10] |   43s | val 2.6052 | test 2.5725
DeepHalo-15           seed 10 | epochs [38, 10] |   50s | val 2.5932 | test 2.5640
DeepHalo-15           seed 11 | epochs [36, 10] |   48s | val 2.6095 | test 2.5819
DeepHalo-15           seed 12 | epochs [30, 10] |   42s | val 2.6190 | test 2.5840
DeepHalo-15           seed 13 | epochs [29, 10] |   41s | val 2.6185 | test 2.5812
DeepHalo-15           seed 14 | epochs [31, 10] |   43s | val 2.6077 | test 2.5731
SelfAttn-w80          seed  0 | epochs [23, 10] |  

In [33]:
# Expedia E10 — What-if: make ONE competitor 10% cheaper; how do the OTHER hotels' scores move, and does it depend on similarity?
#   Context-free models / IIA: every other hotel's score is unchanged -> centred change = 0 everywhere.
#   Similarity effect: hotels SIMILAR to the competitor lose MORE  ->  centred change rises with distance (corr > 0).
CKD = "/kaggle/working/mtp_ckpt/models"
def get_model(name, seed):
    path = f"{CKD}/expedia_{name}_seed{seed}.pt"
    m = EX_MODELS[name]().to(dev)
    if os.path.exists(path): m.load_state_dict(torch.load(path, map_location=dev))
    else: torch.manual_seed(seed); m, _ = ex_train(m, seed=seed)
    return m.eval()

xi, xs, pos, mk, yy = ex_S["test"]
B, N, D = xi.shape
ar = torch.arange(B, device=dev)
j_price = EX_ITEM.index("log_price")
delta = np.log(0.9) / sd_i[ci.index(j_price)] if "sd_i" in globals() else -0.25   # 10% cut in scaled units
print(f"price change applied: {delta:+.3f} (scaled units) = 10% cheaper")

c = torch.multinomial(mk.float().cpu(), 1, generator=torch.Generator().manual_seed(0)).squeeze(1).to(dev)  # one real competitor per slate
xi_cf = xi.clone(); xi_cf[ar, c, j_price] += delta
others = mk.clone(); others[ar, c] = False

xc = xi[ar, c]
nonprice = [j for j in range(D) if j != j_price]
dist_feat  = (xi[:, :, nonprice] - xc[:, None, nonprice]).norm(dim=-1)   # distance on everything except price
dist_price = (xi[:, :, j_price] - xc[:, None, j_price]).abs()            # distance in (original) price

def centred_change(model):
    with torch.no_grad():
        d = model(xi_cf, xs, pos, mk) - model(xi, xs, pos, mk)
    d_c = d[ar, c]                                                    # competitor's own score change
    d = d.masked_fill(~others, 0.0)
    d = d - d.sum(1, keepdim=True) / others.sum(1, keepdim=True)       # remove the slate-wide shift (softmax ignores it)
    return d.masked_fill(~others, 0.0), d_c

def within_corr(a, b, m):                                             # correlation after removing each slate's mean
    n = m.sum(1, keepdim=True)
    a = a.masked_fill(~m, 0); b = b.masked_fill(~m, 0)
    a = (a - a.sum(1, keepdim=True) / n).masked_fill(~m, 0); b = (b - b.sum(1, keepdim=True) / n).masked_fill(~m, 0)
    return ((a * b).sum() / torch.sqrt((a * a).sum() * (b * b).sum())).item()

def within_quintile(dist, m):                                         # 0 = most similar fifth of the slate, 4 = least
    rk = dist.masked_fill(~m, float("inf")).argsort(1).argsort(1).float()
    return (rk / (m.sum(1, keepdim=True).float() - 1).clamp_min(1) * 5).clamp(max=4.999).floor()
q_feat, q_price = within_quintile(dist_feat, others), within_quintile(dist_price, others)

rows_cf, qt_feat, qt_price = [], [], []
for name in ["SelfAttn", "SelfAttn-selfonly"]:
    for seed in range(5):
        dchg, d_c = centred_change(get_model(name, seed))
        rows_cf.append(dict(model=name, seed=seed,
                            competitor_gain=d_c.mean().item(),
                            spread_of_others=dchg[others].std().item(),
                            corr_with_feat_distance=within_corr(dchg, dist_feat, others),
                            corr_with_price_distance=within_corr(dchg, dist_price, others)))
        if name == "SelfAttn":
            qt_feat.append([dchg[others & (q_feat == k)].mean().item() * 100 for k in range(5)])
            qt_price.append([dchg[others & (q_price == k)].mean().item() * 100 for k in range(5)])

print("\nPer model/seed:"); print(pd.DataFrame(rows_cf).round(4).to_string(index=False))
cols = ["most similar", "2", "3", "4", "least similar"]
print("\nSelfAttn: mean centred score change of the other hotels (×100), by similarity to the competitor (5 seeds avg):")
print(pd.DataFrame([np.mean(qt_feat, 0), np.mean(qt_price, 0)], index=["by non-price features", "by price"], columns=cols).round(3))

price change applied: -0.177 (scaled units) = 10% cheaper

Per model/seed:
            model  seed  competitor_gain  spread_of_others  corr_with_feat_distance  corr_with_price_distance
         SelfAttn     0           0.2191            0.0099                   0.1336                    0.3574
         SelfAttn     1           0.1875            0.0088                   0.1940                    0.3394
         SelfAttn     2           0.1970            0.0094                   0.1204                    0.3323
         SelfAttn     3           0.2188            0.0085                   0.0947                    0.2465
         SelfAttn     4           0.1830            0.0085                   0.2376                    0.4191
SelfAttn-selfonly     0           0.1876            0.0000                      NaN                       NaN
SelfAttn-selfonly     1           0.1590            0.0000                      NaN                       NaN
SelfAttn-selfonly     2           0.1752     

In [34]:
# Expedia E11 — Control: same what-if test on models that only see slate AVERAGES (should NOT target similar hotels)
rows_ctrl = []
for name, seeds in [("Pool-mean+sd", range(5)), ("LCL-excl", range(1))]:
    for seed in seeds:
        dchg, d_c = centred_change(get_model(name, seed))
        rows_ctrl.append(dict(model=name, seed=seed, competitor_gain=d_c.mean().item(),
                              spread_of_others=dchg[others].std().item(),
                              corr_with_feat_distance=within_corr(dchg, dist_feat, others),
                              corr_with_price_distance=within_corr(dchg, dist_price, others)))
sa = pd.DataFrame(rows_cf); sa = sa[sa.model == "SelfAttn"]
print(pd.DataFrame(rows_ctrl).round(4).to_string(index=False))
print("\nmeans over seeds:")
print(pd.concat([sa, pd.DataFrame(rows_ctrl)]).groupby("model")[["spread_of_others", "corr_with_feat_distance",
                                                                 "corr_with_price_distance"]].mean().round(4))

       model  seed  competitor_gain  spread_of_others  corr_with_feat_distance  corr_with_price_distance
Pool-mean+sd     0           0.1870            0.0035                   0.1132                    0.3979
Pool-mean+sd     1           0.1865            0.0029                   0.1604                    0.4336
Pool-mean+sd     2           0.1958            0.0042                   0.1114                    0.4029
Pool-mean+sd     3           0.2072            0.0041                   0.0891                    0.4426
Pool-mean+sd     4           0.1803            0.0032                   0.1340                    0.4146
    LCL-excl     0           0.1494            0.0020                   0.0240                    0.1110

means over seeds:
              spread_of_others  corr_with_feat_distance  \
model                                                     
LCL-excl                0.0020                   0.0240   
Pool-mean+sd            0.0036                   0.1216   
SelfAttn  

In [35]:
# Expedia E12 — Competitor-specific test: same slate, cut c1's price OR c2's price; compare the other hotels' responses.
#   Average-based models: both cuts move the slate average identically -> R ≈ 0 for every other hotel.
#   Pairwise comparison:  hotels react more to the competitor they are CLOSER to -> corr(R, dist-to-c1 − dist-to-c2) > 0.
g = torch.Generator().manual_seed(1)
w = mk.float().cpu()
c1 = torch.multinomial(w, 1, generator=g).squeeze(1)
w2 = w.clone(); w2[torch.arange(B), c1] = 0
c2 = torch.multinomial(w2, 1, generator=g).squeeze(1)
c1, c2 = c1.to(dev), c2.to(dev)
oth2 = mk.clone(); oth2[ar, c1] = False; oth2[ar, c2] = False

def cut(cidx):
    x = xi.clone(); x[ar, cidx, j_price] += delta; return x
x1, x2 = cut(c1), cut(c2)

def dist_to(cidx, cols):
    return (xi[:, :, cols] - xi[ar, cidx][:, None, cols]).norm(dim=-1)
dd_feat  = dist_to(c1, nonprice)  - dist_to(c2, nonprice)     # > 0 means the hotel is closer to c2 than to c1
dd_price = dist_to(c1, [j_price]) - dist_to(c2, [j_price])

def centre(d, m):
    d = d.masked_fill(~m, 0.0)
    return (d - d.sum(1, keepdim=True) / m.sum(1, keepdim=True)).masked_fill(~m, 0.0)

def specific_response(model):            # R_t = (response to cutting c1) − (response to cutting c2)
    with torch.no_grad():
        R = model(x1, xs, pos, mk) - model(x2, xs, pos, mk)
    return centre(R, oth2)

rows_sp = []
for name, seeds in [("SelfAttn", range(5)), ("Pool-mean+sd", range(5)), ("LCL-excl", range(1)), ("SelfAttn-selfonly", range(1))]:
    for seed in seeds:
        R = specific_response(get_model(name, seed))
        rows_sp.append(dict(model=name, seed=seed, spread_R=R[oth2].std().item(),
                            corr_feat=within_corr(R, dd_feat, oth2) if R[oth2].std() > 0 else np.nan,
                            corr_price=within_corr(R, dd_price, oth2) if R[oth2].std() > 0 else np.nan))
sp = pd.DataFrame(rows_sp)
print(sp.round(4).to_string(index=False))
print("\nmeans:"); print(sp.groupby("model")[["spread_R", "corr_feat", "corr_price"]].mean().round(4))

            model  seed  spread_R  corr_feat  corr_price
         SelfAttn     0    0.0144     0.1468      0.3878
         SelfAttn     1    0.0122     0.2353      0.3709
         SelfAttn     2    0.0132     0.1702      0.3796
         SelfAttn     3    0.0112     0.1060      0.3025
         SelfAttn     4    0.0114     0.3102      0.4629
     Pool-mean+sd     0    0.0045     0.1439      0.5438
     Pool-mean+sd     1    0.0035     0.1698      0.5799
     Pool-mean+sd     2    0.0055     0.1598      0.5234
     Pool-mean+sd     3    0.0051     0.1659      0.5911
     Pool-mean+sd     4    0.0043     0.1463      0.5346
         LCL-excl     0    0.0000     0.0020      0.0098
SelfAttn-selfonly     0    0.0000        NaN         NaN

means:
                   spread_R  corr_feat  corr_price
model                                             
LCL-excl             0.0000     0.0020      0.0098
Pool-mean+sd         0.0046     0.1571      0.5546
SelfAttn             0.0125     0.1937      0.3

In [36]:
# Expedia E13 — Mechanism ladder by prediction: does similarity-weighting or within-slate rank close SelfAttn's gap?
def sim_weighted_others(xi, mask, log_tau):          # each hotel: softmax-weighted mean of OTHER hotels, weight ∝ exp(−dist²/τ)
    d2 = torch.cdist(xi, xi).pow(2)                   # (B, N, N)
    block = ~(mask.unsqueeze(1) & mask.unsqueeze(2)) | torch.eye(xi.shape[1], dtype=torch.bool, device=xi.device)
    w = torch.softmax((-d2 / log_tau.exp()).masked_fill(block, -1e9), dim=-1)
    return w @ xi

def within_rank(xi, mask):                            # percentile rank of each feature among the OTHER hotels (ties = 0.5)
    a, b = xi.unsqueeze(2), xi.unsqueeze(1)           # (B, N, 1, D), (B, 1, N, D)
    valid = (mask.unsqueeze(1) & ~torch.eye(xi.shape[1], dtype=torch.bool, device=xi.device)).unsqueeze(-1)
    below = ((b < a).float() + 0.5 * (b == a).float()) * valid
    return below.sum(2) / valid.sum(2).clamp_min(1)

class ExPoolPlus(nn.Module):
    def __init__(self, kind, hidden=69):
        super().__init__(); self.kind = kind
        self.log_tau = nn.Parameter(torch.zeros(()))  # learned bandwidth (used by Pool-sim only)
        d_in = D_I + D_S + 3 * D_I
        self.f = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, xi, xs, mask):
        mo, so = others_stats(xi, mask)
        extra = sim_weighted_others(xi, mask, self.log_tau) if self.kind == "sim" else within_rank(xi, mask)
        return self.f(torch.cat([ex_input(xi, xs), mo, so, extra], -1)).squeeze(-1)

EX_MODELS["Pool-sim"]  = lambda: ExModel(ExPoolPlus("sim"))
EX_MODELS["Pool-rank"] = lambda: ExModel(ExPoolPlus("rank"))
for k in ["Pool-mean+sd", "Pool-sim", "Pool-rank", "SelfAttn"]:
    print(f"{k:13s} params {sum(p.numel() for p in EX_MODELS[k]().parameters()):,}")

for name in ["Pool-sim", "Pool-rank"]:
    for seed in range(15):
        torch.manual_seed(seed); t0 = time.time()
        m, ep = ex_train(EX_MODELS[name]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=name, seed=seed, val=v, test=t, acc=acc))
        extra = f" | tau {m.core.log_tau.exp().item():.2f}" if name == "Pool-sim" else ""
        print(f"{name:9s} seed {seed:2d} | epochs {ep} | {time.time()-t0:4.0f}s | val {v:.4f} | test {t:.4f}{extra}")

exr = pd.DataFrame(ex_rows)
ladder = ["SelfAttn-selfonly", "Pool-mean", "Pool-mean+sd", "Pool-sim", "Pool-rank", "SelfAttn"]
print("\nTest NLL ladder:"); print(exr.groupby("model")["test"].agg(["mean", "std", "count"]).reindex(ladder).round(4))
for a, b in [("Pool-sim", "Pool-mean+sd"), ("Pool-rank", "Pool-mean+sd"), ("SelfAttn", "Pool-sim"), ("SelfAttn", "Pool-rank")]:
    print(" ", ex_gap(a, b))

Pool-mean+sd  params 9,588
Pool-sim      params 9,565
Pool-rank     params 9,565
SelfAttn      params 9,482
Pool-sim  seed  0 | epochs [25, 10] |    5s | val 2.5935 | test 2.5655 | tau 1.01
Pool-sim  seed  1 | epochs [30, 11] |    6s | val 2.5986 | test 2.5649 | tau 1.03
Pool-sim  seed  2 | epochs [22, 11] |    4s | val 2.6061 | test 2.5701 | tau 0.97
Pool-sim  seed  3 | epochs [29, 10] |    5s | val 2.5965 | test 2.5667 | tau 1.10
Pool-sim  seed  4 | epochs [24, 10] |    5s | val 2.5973 | test 2.5729 | tau 1.03
Pool-sim  seed  5 | epochs [27, 10] |    5s | val 2.6032 | test 2.5682 | tau 1.06
Pool-sim  seed  6 | epochs [32, 10] |    6s | val 2.5856 | test 2.5590 | tau 1.05
Pool-sim  seed  7 | epochs [25, 10] |    5s | val 2.5951 | test 2.5570 | tau 1.04
Pool-sim  seed  8 | epochs [31, 10] |    5s | val 2.5894 | test 2.5780 | tau 1.07
Pool-sim  seed  9 | epochs [27, 10] |    5s | val 2.5973 | test 2.5648 | tau 1.03
Pool-sim  seed 10 | epochs [21, 12] |    4s | val 2.6026 | test 2.5670 |

In [37]:
# Expedia E14 — Which feature's rank matters? Pool-mean+sd + the rank of ONE feature (5 seeds each), size-matched (~9.5k)
def hidden_for(d_in, target=9500):                       # hidden width giving ~target params for a 2-hidden-layer MLP
    return int((-(d_in + 3) + np.sqrt((d_in + 3) ** 2 + 4 * (target - 1))) / 2)

class ExPoolRankSub(nn.Module):
    def __init__(self, cols):
        super().__init__(); self.cols = list(cols)
        d_in = D_I + D_S + 2 * D_I + len(self.cols); h = hidden_for(d_in)
        self.f = nn.Sequential(nn.Linear(d_in, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, xi, xs, mask):
        mo, so = others_stats(xi, mask)
        rk = within_rank(xi, mask)[..., self.cols]
        return self.f(torch.cat([ex_input(xi, xs), mo, so, rk], -1)).squeeze(-1)

rank_feats = ["log_price", "prop_review_score", "prop_starrating", "prop_location_score1", "prop_location_score2",
              "log_dist", "comp_cheaper", "comp_dearer", "promotion_flag"]
groups = {f"rank:{f}": [EX_ITEM.index(f)] for f in rank_feats}
groups["rank:all-but-price"] = [j for j, c in enumerate(EX_ITEM) if c != "log_price"]
for g_name, cols in groups.items():
    EX_MODELS[g_name] = (lambda cols=cols: ExModel(ExPoolRankSub(cols)))
print("params (one feature):", sum(p.numel() for p in EX_MODELS["rank:log_price"]().parameters()),
      "| params (all-but-price):", sum(p.numel() for p in EX_MODELS["rank:all-but-price"]().parameters()))

for g_name in groups:
    for seed in range(5):
        torch.manual_seed(seed)
        m, ep = ex_train(EX_MODELS[g_name]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=g_name, seed=seed, val=v, test=t, acc=acc))
    x = [r["test"] for r in ex_rows if r["model"] == g_name]
    print(f"{g_name:28s} test {np.mean(x):.4f} ± {np.std(x, ddof=1):.4f}")

exr = pd.DataFrame(ex_rows)
base = exr[exr.model == "Pool-mean+sd"]["test"]
full_gain = base.mean() - exr[exr.model == "Pool-rank"]["test"].mean()
out = []
for g_name in groups:
    x = exr[exr.model == g_name]["test"]
    gain = base.mean() - x.mean()
    out.append(dict(group=g_name, test=x.mean(), gain_vs_mean_sd=gain,
                    se=np.sqrt(x.var() / len(x) + base.var() / len(base)), share_of_all_rank_gain=gain / full_gain))
print(f"\nall-feature rank gain (Pool-rank vs Pool-mean+sd) = {full_gain:.4f}")
print(pd.DataFrame(out).sort_values("gain_vs_mean_sd", ascending=False).round(4).to_string(index=False))

params (one feature): 9459 | params (all-but-price): 9495
rank:log_price               test 2.5634 ± 0.0022
rank:prop_review_score       test 2.5658 ± 0.0061
rank:prop_starrating         test 2.5661 ± 0.0072
rank:prop_location_score1    test 2.5634 ± 0.0039
rank:prop_location_score2    test 2.5649 ± 0.0036
rank:log_dist                test 2.5661 ± 0.0046
rank:comp_cheaper            test 2.5660 ± 0.0060
rank:comp_dearer             test 2.5632 ± 0.0032
rank:promotion_flag          test 2.5674 ± 0.0101
rank:all-but-price           test 2.5657 ± 0.0044

all-feature rank gain (Pool-rank vs Pool-mean+sd) = 0.0055
                    group   test  gain_vs_mean_sd     se  share_of_all_rank_gain
         rank:comp_dearer 2.5632           0.0061 0.0019                  1.1078
rank:prop_location_score1 2.5634           0.0059 0.0021                  1.0743
           rank:log_price 2.5634           0.0059 0.0015                  1.0732
rank:prop_location_score2 2.5649           0.0044 0.0020  

In [38]:
# Expedia E15 — Is the "rank" gain just within-slate normalisation? Pool-mean+sd + explicit z-score (x − mean_others)/sd_others
class ExPoolZ(nn.Module):
    def __init__(self):
        super().__init__()
        d_in = D_I + D_S + 3 * D_I; h = hidden_for(d_in)
        self.f = nn.Sequential(nn.Linear(d_in, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, xi, xs, mask):
        mo, so = others_stats(xi, mask)
        z = ((xi - mo) / (so + 0.1)).clamp(-5, 5)          # +0.1 avoids dividing by ~0 when the slate doesn't vary
        return self.f(torch.cat([ex_input(xi, xs), mo, so, z], -1)).squeeze(-1)

EX_MODELS["Pool-z"] = lambda: ExModel(ExPoolZ())
print("params:", sum(p.numel() for p in EX_MODELS["Pool-z"]().parameters()))
for seed in range(15):
    torch.manual_seed(seed)
    m, ep = ex_train(EX_MODELS["Pool-z"]().to(dev), seed=seed)
    v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
    ex_rows.append(dict(model="Pool-z", seed=seed, val=v, test=t, acc=acc))
exr = pd.DataFrame(ex_rows)
print(exr.groupby("model")["test"].agg(["mean", "std", "count"]).reindex(
      ["Pool-mean+sd", "Pool-z", "Pool-rank", "SelfAttn"]).round(4))
for a, b in [("Pool-z", "Pool-mean+sd"), ("Pool-rank", "Pool-z"), ("SelfAttn", "Pool-z")]:
    print(" ", ex_gap(a, b))

params: 9358
                mean     std  count
model                              
Pool-mean+sd  2.5693  0.0046     15
Pool-z        2.5617  0.0053     15
Pool-rank     2.5638  0.0030     15
SelfAttn      2.5584  0.0056     15
  Pool-z − Pool-mean+sd: -0.0076 ± 0.0018
  Pool-rank − Pool-z: +0.0020 ± 0.0016
  SelfAttn − Pool-z: -0.0034 ± 0.0020


In [1]:
# SETUP — first cell of a fresh session: reload saved data, define every model used so far, verify a checkpoint
import os, glob, json, copy, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- 1. locate the uploaded checkpoint folder and reload data ----------
CK = os.path.dirname(glob.glob("/kaggle/input/**/mtp_ckpt/meta.json", recursive=True)[0])
CKD = f"{CK}/models"
meta = json.load(open(f"{CK}/meta.json"))
EX_ITEM, EX_SHARED, MAXN, SPLIT, MODES = meta["EX_ITEM"], meta["EX_SHARED"], meta["MAXN"], meta["SPLIT"], meta["MODES"]
D_I, D_S = len(EX_ITEM), len(EX_SHARED)
SD_LOG_PRICE = 0.595            # train std of log price (recovered from E10: 10% cut = -0.177 scaled units)

L = np.load(f"{CK}/lpmc_data.npz")
lp_masks = {"train": L["is_train"], "val": L["is_val"], "test": L["is_test"]}
S_year = {k: tuple(torch.tensor(a[m]).to(dev) for a in (L["X_item_s"], L["X_shared_s"], L["y"])) for k, m in lp_masks.items()}
S = S_year

E = np.load(f"{CK}/expedia_data.npz")
ex_masks = {"train": E["is_tr"], "val": E["is_va"], "test": E["is_te"]}
ex_S = {k: tuple(torch.tensor(E[a][m]).to(dev) for a in ("Xi", "Xs", "pos", "mask", "y")) for k, m in ex_masks.items()}

rows = pd.read_csv(f"{CK}/lpmc_results.csv").to_dict("records")
ex_rows = pd.read_csv(f"{CK}/expedia_results.csv").to_dict("records")
print(f"data from {CK} | LPMC {[len(v[-1]) for v in S_year.values()]} | Expedia {[len(v[-1]) for v in ex_S.values()]} "
      f"| results: {len(rows)} LPMC + {len(ex_rows)} Expedia runs | {len(os.listdir(CKD))} checkpoints")

# ---------- 2. DeepHalo (authors' FeatureBased.py, unchanged) ----------
def make_valid_mask(n, lengths, device, mode='without'):
    row = torch.arange(n, device=device).unsqueeze(0)
    if mode == 'with':
        return (row < lengths.unsqueeze(1)) | (row == n - 1)
    return row < lengths.unsqueeze(1)

def masked_softmax(scores, lengths):
    valid = make_valid_mask(scores.size(1), lengths, scores.device)
    return F.log_softmax(scores.masked_fill(~valid, float('-inf')), dim=-1)

class NonlinearTransformation(nn.Module):
    def __init__(self, H, embed=128, dropout=0):
        super().__init__()
        self.fc1 = nn.Linear(embed, embed * H); self.fc2 = nn.Linear(embed, embed)
        self.H = H; self.embed = embed; self.dropout = nn.Dropout(dropout); self.enc_norm = nn.LayerNorm(embed)
    def forward(self, X):
        B, n, _ = X.shape
        X = nn.ReLU()(self.fc1(X).view(B, n, self.H, self.embed))
        return self.enc_norm(self.fc2(self.dropout(X)))

class DeepHalo(nn.Module):
    def __init__(self, n, input_dim, H, L, embed=128, dropout=0):
        super().__init__()
        self.basic_encoder = nn.Sequential(
            nn.Linear(input_dim, embed), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed, embed), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed, embed))
        self.enc_norm = nn.LayerNorm(embed)
        self.aggregate_linear = nn.ModuleList([nn.Linear(embed, H) for _ in range(L)])
        self.nonlinear = nn.ModuleList([NonlinearTransformation(H, embed) for _ in range(L)])
        self.H = H; self.embed = embed
        self.final_linear = nn.Linear(embed, 1)
        self.qualinear1 = nn.Linear(embed, embed)   # defined but unused in the released code
        self.qualinear2 = nn.Linear(embed, embed)
    def forward(self, X, lengths):
        B, n, _ = X.shape
        Z = self.enc_norm(self.basic_encoder(X)); X = Z.clone()
        for fc, nt in zip(self.aggregate_linear, self.nonlinear):
            Z_bar = (fc(Z).sum(1) / lengths.unsqueeze(1)).unsqueeze(-1).unsqueeze(1)
            phi = nt(X) * make_valid_mask(n, lengths, X.device).unsqueeze(-1).unsqueeze(-1)
            Z = (phi * Z_bar).sum(2) / self.H + Z
        logits = self.final_linear(Z).squeeze(-1)
        return masked_softmax(logits, lengths), logits

# ---------- 3. LPMC models + trainer ----------
def make_input(xi, xs):
    return torch.cat([xi, xs.unsqueeze(1).expand(-1, xi.shape[1], -1)], dim=-1)

class DeepHaloChoice(nn.Module):
    def __init__(self, **kw):
        super().__init__(); self.net = DeepHalo(**kw)
    def forward(self, xi, xs):
        X = make_input(xi, xs)
        return self.net(X, torch.full((X.shape[0],), X.shape[1], device=X.device))[1]

class ContextFreeMLP(nn.Module):
    def __init__(self, input_dim=25, hidden=128, layers=3):
        super().__init__(); mods, d = [], input_dim
        for _ in range(layers - 1):
            mods += [nn.Linear(d, hidden), nn.ReLU()]; d = hidden
        self.f = nn.Sequential(*mods, nn.Linear(d, 1))
    def forward(self, xi, xs):
        return self.f(make_input(xi, xs)).squeeze(-1)

class SelfAttnChoice(nn.Module):
    def __init__(self, input_dim=25, width=32, heads=2, ffn=64, layers=1):
        super().__init__()
        self.embed = nn.Linear(input_dim, width); self.ln0 = nn.LayerNorm(width)
        self.blocks = nn.ModuleList([nn.ModuleDict(dict(
            attn=nn.MultiheadAttention(width, heads, batch_first=True), ln1=nn.LayerNorm(width),
            ffn=nn.Sequential(nn.Linear(width, ffn), nn.ReLU(), nn.Linear(ffn, width)), ln2=nn.LayerNorm(width)))
            for _ in range(layers)])
        self.out = nn.Linear(width, 1)
    def forward(self, xi, xs):
        h = self.ln0(self.embed(make_input(xi, xs)))
        for b in self.blocks:
            h = b["ln1"](h + b["attn"](h, h, h, need_weights=False)[0]); h = b["ln2"](h + b["ffn"](h))
        return self.out(h).squeeze(-1)

class SelfAttnSelfOnly(SelfAttnChoice):
    def forward(self, xi, xs):
        h = self.ln0(self.embed(make_input(xi, xs)))
        mask = ~torch.eye(h.shape[1], dtype=torch.bool, device=h.device)
        for b in self.blocks:
            h = b["ln1"](h + b["attn"](h, h, h, attn_mask=mask, need_weights=False)[0]); h = b["ln2"](h + b["ffn"](h))
        return self.out(h).squeeze(-1)

def evaluate(model, split, bs=8192):
    xi, xs, yy = S[split]; model.eval(); nll, correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(yy), bs):
            lg = model(xi[i:i+bs], xs[i:i+bs])
            nll += F.cross_entropy(lg, yy[i:i+bs], reduction="sum").item(); correct += (lg.argmax(1) == yy[i:i+bs]).sum().item()
    return nll / len(yy), correct / len(yy)

def train_model(model, phases=((1e-3, 10), (1e-4, 10)), bs=256, max_epochs=300, seed=0):
    g = torch.Generator().manual_seed(seed); xi, xs, yy = S["train"]; n = len(yy)
    best_v, best_state = evaluate(model, "val")[0], copy.deepcopy(model.state_dict()); epochs_run = []
    for lr, patience in phases:
        model.load_state_dict(best_state); opt = torch.optim.Adam(model.parameters(), lr=lr); wait = 0
        for ep in range(max_epochs):
            model.train(); perm = torch.randperm(n, generator=g).to(dev)
            for i in range(0, n, bs):
                b = perm[i:i+bs]; loss = F.cross_entropy(model(xi[b], xs[b]), yy[b])
                opt.zero_grad(); loss.backward(); opt.step()
            v = evaluate(model, "val")[0]
            if v < best_v - 1e-5: best_v, best_state, wait = v, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
                if wait >= patience: break
        epochs_run.append(ep + 1)
    model.load_state_dict(best_state); return model, epochs_run

# ---------- 4. Expedia models + trainer ----------
NEG = -1e9
class PosBias(nn.Module):
    def __init__(self, n_pos=41):
        super().__init__(); self.b = nn.Embedding(n_pos, 1); nn.init.zeros_(self.b.weight)
    def forward(self, pos):
        return self.b(pos).squeeze(-1)

def ex_input(xi, xs):
    return torch.cat([xi, xs.unsqueeze(1).expand(-1, xi.shape[1], -1)], dim=-1)

class ExModel(nn.Module):
    def __init__(self, core):
        super().__init__(); self.core = core; self.pos = PosBias()
    def forward(self, xi, xs, pos, mask):
        return (self.core(xi, xs, mask) + self.pos(pos)).masked_fill(~mask, NEG)

class ExZero(nn.Module):
    def forward(self, xi, xs, mask):
        return torch.zeros(mask.shape, device=mask.device)

class ExMNL_A(nn.Module):
    def __init__(self, d_i=D_I):
        super().__init__(); self.beta = nn.Linear(d_i, 1, bias=False)
    def forward(self, xi, xs, mask):
        return self.beta(xi).squeeze(-1)

class ExMNL_B(ExMNL_A):
    def __init__(self, d_i=D_I, d_s=D_S):
        super().__init__(d_i); self.G = nn.Parameter(torch.zeros(d_i, d_s))
    def forward(self, xi, xs, mask):
        return super().forward(xi, xs, mask) + torch.einsum("bnd,de,be->bn", xi, self.G, xs)

def mlp(d_in, h):
    return nn.Sequential(nn.Linear(d_in, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))

def hidden_for(d_in, target=9500):
    return int((-(d_in + 3) + np.sqrt((d_in + 3) ** 2 + 4 * (target - 1))) / 2)

class ExMLP(nn.Module):
    def __init__(self, d_in=D_I + D_S, hidden=85):
        super().__init__(); self.f = mlp(d_in, hidden)
    def forward(self, xi, xs, mask):
        return self.f(ex_input(xi, xs)).squeeze(-1)

class ExDeepHalo(nn.Module):           # authors' layers; padded hotels zeroed before every aggregation
    def __init__(self, d_in=D_I + D_S, H=8, L=4, embed=15):
        super().__init__(); self.net = DeepHalo(n=MAXN, input_dim=d_in, H=H, L=L, embed=embed)
    def forward(self, xi, xs, mask):
        net, m = self.net, mask.unsqueeze(-1).float(); lengths = mask.sum(1).float()
        Z = net.enc_norm(net.basic_encoder(ex_input(xi, xs))) * m; X = Z.clone()
        for fc, nt in zip(net.aggregate_linear, net.nonlinear):
            Z_bar = ((fc(Z) * m).sum(1) / lengths.unsqueeze(1)).unsqueeze(-1).unsqueeze(1)
            Z = (((nt(X) * m.unsqueeze(-1)) * Z_bar).sum(2) / net.H + Z) * m
        return net.final_linear(Z).squeeze(-1)

class ExSelfAttn(nn.Module):
    def __init__(self, d_in=D_I + D_S, width=32, heads=2, ffn=64, self_only=False):
        super().__init__(); self.self_only = self_only
        self.embed = nn.Linear(d_in, width); self.ln0 = nn.LayerNorm(width)
        self.attn = nn.MultiheadAttention(width, heads, batch_first=True); self.ln1 = nn.LayerNorm(width)
        self.ffn = nn.Sequential(nn.Linear(width, ffn), nn.ReLU(), nn.Linear(ffn, width)); self.ln2 = nn.LayerNorm(width)
        self.out = nn.Linear(width, 1)
    def forward(self, xi, xs, mask):
        h = self.ln0(self.embed(ex_input(xi, xs)))
        if self.self_only:
            blocked = ~torch.eye(h.shape[1], dtype=torch.bool, device=h.device)
            a = self.attn(h, h, h, attn_mask=blocked, need_weights=False)[0]
        else:
            a = self.attn(h, h, h, key_padding_mask=~mask, need_weights=False)[0]
        h = self.ln1(h + a); h = self.ln2(h + self.ffn(h))
        return self.out(h).squeeze(-1)

def others_stats(xi, mask):
    m = mask.unsqueeze(-1).float(); n = m.sum(1, keepdim=True)
    s1 = (xi * m).sum(1, keepdim=True); s2 = (xi ** 2 * m).sum(1, keepdim=True)
    mean_o = (s1 - xi) / (n - 1)
    return mean_o, ((s2 - xi ** 2) / (n - 1) - mean_o ** 2).clamp_min(0).sqrt()

def sim_weighted_others(xi, mask, log_tau):
    d2 = torch.cdist(xi, xi).pow(2)
    block = ~(mask.unsqueeze(1) & mask.unsqueeze(2)) | torch.eye(xi.shape[1], dtype=torch.bool, device=xi.device)
    return torch.softmax((-d2 / log_tau.exp()).masked_fill(block, -1e9), dim=-1) @ xi

def within_rank(xi, mask):
    a, b = xi.unsqueeze(2), xi.unsqueeze(1)
    valid = (mask.unsqueeze(1) & ~torch.eye(xi.shape[1], dtype=torch.bool, device=xi.device)).unsqueeze(-1)
    return (((b < a).float() + 0.5 * (b == a).float()) * valid).sum(2) / valid.sum(2).clamp_min(1)

class ExLCL(nn.Module):
    def __init__(self, d_i=D_I):
        super().__init__(); self.theta = nn.Linear(d_i, 1, bias=False); self.A = nn.Parameter(torch.zeros(d_i, d_i))
    def forward(self, xi, xs, mask):
        m = mask.unsqueeze(-1).float(); ybar = (xi * m).sum(1) / m.sum(1)
        return self.theta(xi).squeeze(-1) + torch.einsum("bnd,de,be->bn", xi, self.A, ybar)

class ExLCLx(ExLCL):
    def forward(self, xi, xs, mask):
        mo, _ = others_stats(xi, mask)
        return self.theta(xi).squeeze(-1) + torch.einsum("bnd,de,bnd->bn", xi, self.A, mo)

class ExPool(nn.Module):
    def __init__(self, use_sd=False, hidden=79):
        super().__init__(); self.use_sd = use_sd
        self.f = mlp(D_I + D_S + D_I * (2 if use_sd else 1), hidden)
    def forward(self, xi, xs, mask):
        mo, so = others_stats(xi, mask)
        return self.f(torch.cat([ex_input(xi, xs), mo] + ([so] if self.use_sd else []), -1)).squeeze(-1)

class ExPoolPlus(nn.Module):
    def __init__(self, kind, hidden=69):
        super().__init__(); self.kind = kind; self.log_tau = nn.Parameter(torch.zeros(()))
        self.f = mlp(D_I + D_S + 3 * D_I, hidden)
    def forward(self, xi, xs, mask):
        mo, so = others_stats(xi, mask)
        extra = sim_weighted_others(xi, mask, self.log_tau) if self.kind == "sim" else within_rank(xi, mask)
        return self.f(torch.cat([ex_input(xi, xs), mo, so, extra], -1)).squeeze(-1)

class ExPoolZ(nn.Module):
    def __init__(self):
        super().__init__(); d_in = D_I + D_S + 3 * D_I; self.f = mlp(d_in, hidden_for(d_in))
    def forward(self, xi, xs, mask):
        mo, so = others_stats(xi, mask)
        z = ((xi - mo) / (so + 0.1)).clamp(-5, 5)
        return self.f(torch.cat([ex_input(xi, xs), mo, so, z], -1)).squeeze(-1)

EX_MODELS = {
    "Position-only":         lambda: ExModel(ExZero()),
    "MNL-A":                 lambda: ExModel(ExMNL_A()),
    "MNL-B":                 lambda: ExModel(ExMNL_B()),
    "LCL":                   lambda: ExModel(ExLCL()),
    "LCL-excl":              lambda: ExModel(ExLCLx()),
    "MLP-85":                lambda: ExModel(ExMLP(hidden=85)),
    "DeepHalo-15":           lambda: ExModel(ExDeepHalo(embed=15)),
    "DeepHalo-paper":        lambda: ExModel(ExDeepHalo(embed=32, L=5)),
    "SelfAttn":              lambda: ExModel(ExSelfAttn()),
    "SelfAttn-selfonly":     lambda: ExModel(ExSelfAttn(self_only=True)),
    "SelfAttn-w80":          lambda: ExModel(ExSelfAttn(width=80, ffn=160)),
    "SelfAttn-w80-selfonly": lambda: ExModel(ExSelfAttn(width=80, ffn=160, self_only=True)),
    "Pool-mean":             lambda: ExModel(ExPool(use_sd=False, hidden=79)),
    "Pool-mean+sd":          lambda: ExModel(ExPool(use_sd=True, hidden=74)),
    "Pool-sim":              lambda: ExModel(ExPoolPlus("sim")),
    "Pool-rank":             lambda: ExModel(ExPoolPlus("rank")),
    "Pool-z":                lambda: ExModel(ExPoolZ()),
}

def ex_evaluate(model, split, bs=1024):
    xi, xs, pos, mk, yy = ex_S[split]; model.eval(); nll, top1 = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(yy), bs):
            lg = model(xi[i:i+bs], xs[i:i+bs], pos[i:i+bs], mk[i:i+bs])
            nll += F.cross_entropy(lg, yy[i:i+bs], reduction="sum").item(); top1 += (lg.argmax(1) == yy[i:i+bs]).sum().item()
    return nll / len(yy), top1 / len(yy)

def ex_train(model, phases=((1e-3, 10), (1e-4, 10)), bs=256, max_epochs=300, seed=0):
    g = torch.Generator().manual_seed(seed); xi, xs, pos, mk, yy = ex_S["train"]; n = len(yy)
    best_v, best_state = ex_evaluate(model, "val")[0], copy.deepcopy(model.state_dict()); epochs_run = []
    for lr, patience in phases:
        model.load_state_dict(best_state); opt = torch.optim.Adam(model.parameters(), lr=lr); wait = 0
        for ep in range(max_epochs):
            model.train(); perm = torch.randperm(n, generator=g).to(dev)
            for i in range(0, n, bs):
                b = perm[i:i+bs]; loss = F.cross_entropy(model(xi[b], xs[b], pos[b], mk[b]), yy[b])
                opt.zero_grad(); loss.backward(); opt.step()
            v = ex_evaluate(model, "val")[0]
            if v < best_v - 1e-5: best_v, best_state, wait = v, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
                if wait >= patience: break
        epochs_run.append(ep + 1)
    model.load_state_dict(best_state); return model, epochs_run

def ex_gap(a, b):
    exr = pd.DataFrame(ex_rows); x, y = exr[exr.model == a]["test"], exr[exr.model == b]["test"]
    return f"{a} − {b}: {x.mean() - y.mean():+.4f} ± {np.sqrt(x.var()/len(x) + y.var()/len(y)):.4f}"

def get_model(name, seed):            # load a saved checkpoint if there is one, otherwise train it
    path = f"{CKD}/expedia_{name}_seed{seed}.pt"
    m = EX_MODELS[name]().to(dev)
    if os.path.exists(path): m.load_state_dict(torch.load(path, map_location=dev, weights_only=True))
    else: torch.manual_seed(seed); m, _ = ex_train(m, seed=seed)
    return m.eval()

# ---------- 5. verify: saved checkpoints load and reproduce their test scores ----------
for name, seed in [("SelfAttn", 0), ("SelfAttn-selfonly", 0), ("DeepHalo-15", 0)]:
    print(f"{name:17s} seed {seed}: test NLL {ex_evaluate(get_model(name, seed), 'test')[0]:.4f}")
m = SelfAttnChoice().to(dev); m.load_state_dict(torch.load(f"{CKD}/lpmc_SelfAttn_seed0.pt", map_location=dev, weights_only=True))
print(f"LPMC SelfAttn      seed 0: test NLL {evaluate(m, 'test')[0]:.4f}")

data from /kaggle/input/datasets/ajazhussainda25m006/mtp-ckpts/mtp_ckpt | LPMC [49327, 5439, 26320] | Expedia [11092, 1584, 3169] | results: 65 LPMC + 215 Expedia runs | 22 checkpoints
SelfAttn          seed 0: test NLL 2.5594
SelfAttn-selfonly seed 0: test NLL 2.5790
DeepHalo-15       seed 0: test NLL 2.5719
LPMC SelfAttn      seed 0: test NLL 0.6766


In [2]:
# A1+A2 — DeepHalo's best shot on our Expedia data: size grid (paper's + smaller) × {linear, quadratic}, seed 0, select by VAL
class ExDeepHaloQ(ExDeepHalo):
    """Quadratic variant (our reconstruction of the paper's 'Hadamard product' activation, eq. 6):
       aggregate W·σ(z_k) with σ(z) = (W1 z) ⊙ (W2 z), using the qualinear1/2 layers the released code defines but never uses."""
    def forward(self, xi, xs, mask):
        net, m = self.net, mask.unsqueeze(-1).float(); lengths = mask.sum(1).float()
        Z = net.enc_norm(net.basic_encoder(ex_input(xi, xs))) * m; X = Z.clone()
        for fc, nt in zip(net.aggregate_linear, net.nonlinear):
            sig = net.qualinear1(Z) * net.qualinear2(Z)
            Z_bar = ((fc(sig) * m).sum(1) / lengths.unsqueeze(1)).unsqueeze(-1).unsqueeze(1)
            Z = (((nt(X) * m.unsqueeze(-1)) * Z_bar).sum(2) / net.H + Z) * m
        return net.final_linear(Z).squeeze(-1)

grid = [(e, L, H) for e in (32, 64) for L in (4, 5) for H in (8, 16)] + [(e, L, 8) for e in (8, 15) for L in (2, 4)]
variants = {"lin": ExDeepHalo, "quad": ExDeepHaloQ}
dh_grid = []
for vname, cls in variants.items():
    for e, L, H in grid:
        name = f"DH-{vname}-e{e}-L{L}-H{H}"
        EX_MODELS[name] = (lambda cls=cls, e=e, L=L, H=H: ExModel(cls(embed=e, L=L, H=H)))
        torch.manual_seed(0); t0 = time.time()
        m = EX_MODELS[name]().to(dev)
        n_par = sum(p.numel() for n_, p in m.named_parameters() if vname == "quad" or "qualinear" not in n_)
        m, ep = ex_train(m, seed=0)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        dh_grid.append(dict(variant=vname, embed=e, L=L, H=H, params=n_par, epochs=str(ep), secs=round(time.time()-t0),
                            val=v, test=t, nan=bool(np.isnan(v))))
        ex_rows.append(dict(model=name, seed=0, val=v, test=t, acc=acc))
        print(f"{name:20s} params {n_par:7,} | epochs {ep} | {time.time()-t0:4.0f}s | val {v:.4f} | test {t:.4f}")

g = pd.DataFrame(dh_grid)
print("\nRanked by VALIDATION (selection uses val only):")
print(g.sort_values("val").round(4).to_string(index=False))
best = g.loc[g.groupby("variant")["val"].idxmin()]
print("\nBest per variant (by val):"); print(best[["variant", "embed", "L", "H", "params", "val", "test"]].round(4).to_string(index=False))
print("\nreference (15 seeds): SelfAttn test 2.5584 | DeepHalo-15 (lin) 2.5783 | MLP-85 2.5791")

DH-lin-e32-L4-H8     params  42,378 | epochs [20, 11] |   22s | val 2.6149 | test 2.6037
DH-lin-e32-L4-H16    params  77,226 | epochs [20, 10] |   33s | val 2.6233 | test 2.6103
DH-lin-e32-L5-H8     params  52,210 | epochs [20, 10] |   23s | val 2.6141 | test 2.5984
DH-lin-e32-L5-H16    params  95,770 | epochs [20, 10] |   43s | val 2.6343 | test 2.6109
DH-lin-e64-L4-H8     params 162,506 | epochs [13, 12] |   25s | val 2.6524 | test 2.6256
DH-lin-e64-L4-H16    params 297,706 | epochs [17, 10] |   51s | val 2.6528 | test 2.6236
DH-lin-e64-L5-H8     params 200,594 | epochs [17, 10] |   34s | val 2.6374 | test 2.6242
DH-lin-e64-L5-H16    params 369,594 | epochs [13, 10] |   54s | val 2.6506 | test 2.6330
DH-lin-e8-L2-H8      params   1,882 | epochs [53, 10] |   15s | val 2.5970 | test 2.5717
DH-lin-e8-L4-H8      params   3,354 | epochs [37, 10] |   20s | val 2.5917 | test 2.5698
DH-lin-e15-L2-H8     params   5,578 | epochs [36, 11] |   26s | val 2.6076 | test 2.5850
DH-lin-e15-L4-H8     

In [3]:
# A1b — Extend DeepHalo's grid below embed 8, then 15 seeds for every config within 0.003 of the best VALIDATION score
ext = [(e, L, 8) for e in (4, 6) for L in (2, 4)]
for vname, cls in variants.items():
    for e, L, H in ext:
        name = f"DH-{vname}-e{e}-L{L}-H{H}"
        EX_MODELS[name] = (lambda cls=cls, e=e, L=L, H=H: ExModel(cls(embed=e, L=L, H=H)))
        torch.manual_seed(0); t0 = time.time()
        m = EX_MODELS[name]().to(dev)
        n_par = sum(p.numel() for n_, p in m.named_parameters() if vname == "quad" or "qualinear" not in n_)
        m, ep = ex_train(m, seed=0)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        dh_grid.append(dict(variant=vname, embed=e, L=L, H=H, params=n_par, epochs=str(ep), secs=round(time.time()-t0),
                            val=v, test=t, nan=bool(np.isnan(v))))
        ex_rows.append(dict(model=name, seed=0, val=v, test=t, acc=acc))
        print(f"{name:20s} params {n_par:6,} | val {v:.4f} | test {t:.4f}")

g = pd.DataFrame(dh_grid)
finalists = g[g.val <= g.val.min() + 0.003].sort_values("val")
print("\nFinalists (within 0.003 of best val):"); print(finalists[["variant", "embed", "L", "H", "params", "val", "test"]].round(4).to_string(index=False))

for _, r in finalists.iterrows():
    name = f"DH-{r.variant}-e{r.embed}-L{r.L}-H{r.H}"
    for seed in range(1, 15):                          # seed 0 already done
        torch.manual_seed(seed)
        m, ep = ex_train(EX_MODELS[name]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=name, seed=seed, val=v, test=t, acc=acc))
    x = pd.DataFrame(ex_rows); x = x[x.model == name]
    print(f"{name:20s} 15 seeds | val {x.val.mean():.4f} | test {x.test.mean():.4f} ± {x.test.std():.4f}")

print("\nHead-to-head with SelfAttn (untuned thesis model, 15 seeds):")
for _, r in finalists.iterrows():
    print(" ", ex_gap("SelfAttn", f"DH-{r.variant}-e{r.embed}-L{r.L}-H{r.H}"))

DH-lin-e4-L2-H8      params    650 | val 2.6010 | test 2.5696
DH-lin-e4-L4-H8      params  1,106 | val 2.6007 | test 2.5675
DH-lin-e6-L2-H8      params  1,186 | val 2.5970 | test 2.5670
DH-lin-e6-L4-H8      params  2,078 | val 2.5854 | test 2.5557
DH-quad-e4-L2-H8     params    690 | val 2.5809 | test 2.5612
DH-quad-e4-L4-H8     params  1,146 | val 2.5922 | test 2.5690
DH-quad-e6-L2-H8     params  1,270 | val 2.5958 | test 2.5664
DH-quad-e6-L4-H8     params  2,162 | val 2.5865 | test 2.5578

Finalists (within 0.003 of best val):
variant  embed  L  H  params    val   test
   quad      4  2  8     690 2.5809 2.5612
DH-quad-e4-L2-H8     15 seeds | val 2.5999 | test 2.5683 ± 0.0078

Head-to-head with SelfAttn (untuned thesis model, 15 seeds):
  SelfAttn − DH-quad-e4-L2-H8: -0.0099 ± 0.0025


In [4]:
# A1c — Fair selection: top 6 configs (by seed-0 val) get 5 seeds; pick by MEAN val; complete the winner to 15 seeds
g = pd.DataFrame(dh_grid)
top6 = g.sort_values("val").head(6)
sel = []
for _, r in top6.iterrows():
    name = f"DH-{r.variant}-e{r.embed}-L{r.L}-H{r.H}"
    have = {row["seed"] for row in ex_rows if row["model"] == name}
    for seed in range(5):
        if seed in have: continue
        torch.manual_seed(seed)
        m, _ = ex_train(EX_MODELS[name]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=name, seed=seed, val=v, test=t, acc=acc))
    x = pd.DataFrame(ex_rows); x = x[(x.model == name) & (x.seed < 5)]
    sel.append(dict(model=name, params=r.params, val5=x.val.mean(), val5_sd=x.val.std()))
    print(f"{name:20s} params {r.params:6,} | mean val (5 seeds) {x.val.mean():.4f} ± {x.val.std():.4f}")

winner = min(sel, key=lambda d: d["val5"])["model"]
print(f"\nselected by mean val: {winner}")
have = {row["seed"] for row in ex_rows if row["model"] == winner}
for seed in range(15):
    if seed in have: continue
    torch.manual_seed(seed)
    m, _ = ex_train(EX_MODELS[winner]().to(dev), seed=seed)
    v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
    ex_rows.append(dict(model=winner, seed=seed, val=v, test=t, acc=acc))

exr = pd.DataFrame(ex_rows)
for nm in [winner, "SelfAttn"]:
    x = exr[exr.model == nm]
    print(f"{nm:20s} n={len(x):2d} | val {x.val.mean():.4f} | test {x.test.mean():.4f} ± {x.test.std():.4f}")
print(" ", ex_gap("SelfAttn", winner))

DH-quad-e4-L2-H8     params    690 | mean val (5 seeds) 2.5970 ± 0.0111
DH-lin-e6-L4-H8      params  2,078 | mean val (5 seeds) 2.5935 ± 0.0048
DH-quad-e6-L4-H8     params  2,162 | mean val (5 seeds) 2.5906 ± 0.0025
DH-lin-e8-L4-H8      params  3,354 | mean val (5 seeds) 2.5977 ± 0.0052
DH-quad-e4-L4-H8     params  1,146 | mean val (5 seeds) 2.5964 ± 0.0088
DH-quad-e8-L2-H8     params  2,026 | mean val (5 seeds) 2.5965 ± 0.0103

selected by mean val: DH-quad-e6-L4-H8
DH-quad-e6-L4-H8     n=15 | val 2.5908 | test 2.5585 ± 0.0072
SelfAttn             n=15 | val 2.5829 | test 2.5584 ± 0.0056
  SelfAttn − DH-quad-e6-L4-H8: -0.0001 ± 0.0024


In [5]:
# A2 — (1) DeepHalo's blindfolded twin at its tuned size; (2) shrink self-attention, select by 5-seed mean val, then 15 seeds
class ExDeepHaloQSelf(ExDeepHaloQ):            # same params; each hotel's "slate summary" = its OWN aggregate (no cross-hotel info)
    def forward(self, xi, xs, mask):
        net, m = self.net, mask.unsqueeze(-1).float()
        Z = net.enc_norm(net.basic_encoder(ex_input(xi, xs))) * m; X = Z.clone()
        for fc, nt in zip(net.aggregate_linear, net.nonlinear):
            Z_self = fc(net.qualinear1(Z) * net.qualinear2(Z)).unsqueeze(-1)       # (B, n, H, 1) instead of slate mean
            Z = (((nt(X) * m.unsqueeze(-1)) * Z_self).sum(2) / net.H + Z) * m
        return net.final_linear(Z).squeeze(-1)

EX_MODELS["DH-quad-e6-L4-H8-selfonly"] = lambda: ExModel(ExDeepHaloQSelf(embed=6, L=4, H=8))
for seed in range(15):
    torch.manual_seed(seed)
    m, _ = ex_train(EX_MODELS["DH-quad-e6-L4-H8-selfonly"]().to(dev), seed=seed)
    v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
    ex_rows.append(dict(model="DH-quad-e6-L4-H8-selfonly", seed=seed, val=v, test=t, acc=acc))
print(" ", ex_gap("DH-quad-e6-L4-H8", "DH-quad-e6-L4-H8-selfonly"), "  <- DeepHalo's context gain")
print(" ", ex_gap("SelfAttn", "SelfAttn-selfonly"), "  <- self-attention's context gain")

# ---- shrink self-attention: candidates, 5 seeds each, select by mean val
sa_grid = {"SA-w8": dict(width=8, ffn=16), "SA-w16": dict(width=16, ffn=32), "SA-w24": dict(width=24, ffn=48)}
sel = []
for nm, kw in sa_grid.items():
    EX_MODELS[nm] = (lambda kw=kw: ExModel(ExSelfAttn(**kw)))
    EX_MODELS[nm + "-selfonly"] = (lambda kw=kw: ExModel(ExSelfAttn(self_only=True, **kw)))
    for seed in range(5):
        torch.manual_seed(seed)
        m, _ = ex_train(EX_MODELS[nm]().to(dev), seed=seed)
        v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
        ex_rows.append(dict(model=nm, seed=seed, val=v, test=t, acc=acc))
    x = pd.DataFrame(ex_rows); x = x[x.model == nm]
    sel.append((x.val.mean(), nm))
    print(f"{nm:7s} params {sum(p.numel() for p in EX_MODELS[nm]().parameters()):5,} | mean val (5 seeds) {x.val.mean():.4f} ± {x.val.std():.4f}")
x = pd.DataFrame(ex_rows); x = x[(x.model == "SelfAttn") & (x.seed < 5)]
sel.append((x.val.mean(), "SelfAttn")); print(f"SelfAttn params 9,482 | mean val (5 seeds) {x.val.mean():.4f}")

best_sa = min(sel)[1]; print(f"\nselected self-attention size: {best_sa}")
if best_sa != "SelfAttn":
    for nm in [best_sa, best_sa + "-selfonly"]:
        have = {r["seed"] for r in ex_rows if r["model"] == nm}
        for seed in range(15):
            if seed in have: continue
            torch.manual_seed(seed)
            m, _ = ex_train(EX_MODELS[nm]().to(dev), seed=seed)
            v, (t, acc) = ex_evaluate(m, "val")[0], ex_evaluate(m, "test")
            ex_rows.append(dict(model=nm, seed=seed, val=v, test=t, acc=acc))
    print(" ", ex_gap(best_sa, best_sa + "-selfonly"), "  <- context gain at the tuned size")
print(" ", ex_gap(best_sa, "DH-quad-e6-L4-H8"), "  <- tuned vs tuned")

  DH-quad-e6-L4-H8 − DH-quad-e6-L4-H8-selfonly: -0.0092 ± 0.0023   <- DeepHalo's context gain
  SelfAttn − SelfAttn-selfonly: -0.0177 ± 0.0017   <- self-attention's context gain
SA-w8   params   866 | mean val (5 seeds) 2.5843 ± 0.0090
SA-w16  params 2,714 | mean val (5 seeds) 2.5760 ± 0.0060
SA-w24  params 5,586 | mean val (5 seeds) 2.5745 ± 0.0073
SelfAttn params 9,482 | mean val (5 seeds) 2.5810

selected self-attention size: SA-w24
  SA-w24 − SA-w24-selfonly: -0.0169 ± 0.0015   <- context gain at the tuned size
  SA-w24 − DH-quad-e6-L4-H8: -0.0032 ± 0.0023   <- tuned vs tuned


In [7]:
import os, json, zipfile, torch, pandas as pd

OUT = "/kaggle/working/mtp_ckpt2"
os.makedirs(OUT + "/models", exist_ok=True)

# 1) Result tables (everything run since the setup cell: DeepHalo grid, SA shrink, twins)
if "ex_rows" in globals():
    pd.DataFrame(ex_rows).to_csv(f"{OUT}/expedia_results_all.csv", index=False)
    print("expedia rows:", len(ex_rows))
for name in ["lpmc_rows", "rows"]:
    if name in globals() and isinstance(globals()[name], list):
        pd.DataFrame(globals()[name]).to_csv(f"{OUT}/{name}_all.csv", index=False)
        print(name, "rows:", len(globals()[name]))

# 2) Any other result dataframes / dicts you created (A1, A1b, A1c, A2 tables)
for k, v in list(globals().items()):
    if k.startswith("_"): continue
    if isinstance(v, pd.DataFrame) and k not in ("df",) and len(v) < 5000:
        v.to_csv(f"{OUT}/df_{k}.csv", index=False)
        print("saved df:", k, v.shape)

# 3) Trained model weights, if you kept them in a dict (name -> nn.Module)
n_saved = 0
for k, v in list(globals().items()):
    if isinstance(v, dict) and v and all(isinstance(m, torch.nn.Module) for m in v.values()):
        for mk, m in v.items():
            torch.save(m.state_dict(), f"{OUT}/models/{k}__{mk}.pt"); n_saved += 1
    elif isinstance(v, torch.nn.Module) and not k.startswith("_"):
        torch.save(v.state_dict(), f"{OUT}/models/{k}.pt"); n_saved += 1
print("model files saved:", n_saved)

# 4) Small note on what this snapshot contains
json.dump({"note": "after A1c/A2: DeepHalo tuned grid, SA shrink grid, blindfolded twins",
           "base_dataset": "mtp_ckpt"}, open(f"{OUT}/meta2.json", "w"))

# 5) Zip for download
zp = "/kaggle/working/mtp_ckpt2.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            p = os.path.join(root, f); z.write(p, os.path.relpath(p, OUT))
print("zip ready:", zp, round(os.path.getsize(zp)/1e6, 2), "MB")

expedia rows: 346
rows rows: 65
saved df: g (32, 10)
saved df: best (2, 10)
saved df: finalists (1, 10)
saved df: x (5, 5)
saved df: top6 (6, 10)
saved df: exr (291, 5)
model files saved: 1
zip ready: /kaggle/working/mtp_ckpt2.zip 0.04 MB
